# Akili CL v0.23T — Final Nonlinear Groupwise Rescue Verification

This is the **single final Phase-2 experiment**.

It reads the persisted v0.23R replay/protected-probe action evidence and performs no image inference, no semantic-bank reconstruction, no checkpoint training, and no official-test evaluation.

The experiment compares six predeclared nonlinear specifications:

- two-stage rescue gate + expert ranker using Extra Trees;
- two-stage rescue gate + expert ranker using Random Forest;
- their probability blend;
- joint groupwise action selection using Extra Trees;
- joint groupwise action selection using Random Forest;
- their probability blend.

All selection is nested leave-one-seed-out on replay evidence. Protected probes remain unopened until nested replay validation and replay-only global OOF calibration both pass.

## Binding decision

After this notebook finishes, Phase 2 closes and work moves to Phase 3.

- Capacity below 79%: close because the current action set is insufficient.
- Capacity at/above 79% but nonlinear replay validation fails: close because the persisted evidence is not practically retrievable.
- Replay passes but protected probes veto: close and redesign verifier supervision in Phase 3.
- Replay and protected probes pass: close Phase 2 with the validated verifier carried into Phase 3.

No further Phase-2 controller experiment follows v0.23T.

In [ ]:
# Single user configuration section. Empty path overrides preserve automatic discovery.
import json
import os

USER_OVERRIDES = {
    "AKILI_V023T_MODE": "full",
    "AKILI_V023T_PROJECT_ROOT": "",
    "AKILI_V023T_SOURCE_RUN_ROOT": "",
    "AKILI_V023T_OUTPUT_SUBDIR": "stage04/v0_23T_final_nonlinear_groupwise_verifier",
    "AKILI_V023T_RESUME": "1",
    "AKILI_V023T_FAIL_ON_HARD_CHECK": "1",
    "AKILI_V023T_TREE_ESTIMATORS": "220",
    "AKILI_V023T_RANDOM_FOREST_ESTIMATORS": "220",
    "AKILI_V023T_N_JOBS": "-1",
}
for key, value in USER_OVERRIDES.items():
    if value not in (None, ""):
        os.environ[key] = str(value)
    else:
        os.environ.pop(key, None)

SKIP_REAL = os.getenv("AKILI_V023T_SKIP_REAL", "0").strip().lower() in {
    "1", "true", "yes", "on"
}
print("Real Drive execution skipped:", SKIP_REAL)
print(json.dumps(USER_OVERRIDES, indent=2))

In [ ]:
# Mount Google Drive only for the real Colab experiment.
if SKIP_REAL:
    print("Drive mount skipped for local verification.")
else:
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError(
            "The real experiment must run in Google Colab or use AKILI_V023T_SKIP_REAL=1."
        ) from error
    drive.mount("/content/drive")

In [ ]:
# Dependency and runtime validation. No package installation is required.
import joblib
import numpy as np
import pandas as pd
import sklearn
import torch

print("Python runtime ready")
print("torch:", torch.__version__)
print("sklearn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

In [ ]:
# Write, compile, and import the complete standalone v0.23T module.
from pathlib import Path
import ast
import hashlib
import importlib
import sys

MODULE_SOURCE = 'from __future__ import annotations\n\nimport ast\nimport dataclasses\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport tempfile\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\nimport torch\nfrom sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\n\nPROTOCOL = "0.23T-final-nonlinear-groupwise-rescue-verification"\nSOURCE_PROTOCOL = "0.23R-reconstructed-action-evidence-baseline"\nEXPECTED_SCHEMA_VERSION = "0.23R-action-schema-v1"\nEXPECTED_FEATURE_COUNT = 26\nCANDIDATE_COUNT = 3\n\nMODEL_SPECS: Mapping[str, Mapping[str, Any]] = {\n    "two_stage_extra_trees": {\n        "mode": "two_stage",\n        "backends": (("extra_trees", 1.0),),\n    },\n    "two_stage_random_forest": {\n        "mode": "two_stage",\n        "backends": (("random_forest", 1.0),),\n    },\n    "two_stage_blend": {\n        "mode": "two_stage",\n        "backends": (\n            ("extra_trees", 0.5),\n            ("random_forest", 0.5),\n        ),\n    },\n    "joint_extra_trees": {\n        "mode": "joint",\n        "backends": (("extra_trees", 1.0),),\n    },\n    "joint_random_forest": {\n        "mode": "joint",\n        "backends": (("random_forest", 1.0),),\n    },\n    "joint_blend": {\n        "mode": "joint",\n        "backends": (\n            ("extra_trees", 0.5),\n            ("random_forest", 0.5),\n        ),\n    },\n}\n\nUTILITY_RESCUE = 1.0\nUTILITY_DAMAGE = -3.0\nUTILITY_WRONG_TO_WRONG = -0.25\nUTILITY_UNCHANGED = 0.0\n\n\nclass ProtocolError(RuntimeError):\n    pass\n\n\ndef finite_float(value: Any, default: float) -> float:\n    """Parse a metric that may have passed through JSON (NaN -> None).\n\n    Returns ``default`` for None, booleans, non-numeric values, NaN, or infinity.\n    This is used only for ranking/guard logic; raw metric payloads retain their\n    original undefined-precision representation.\n    """\n    if value is None or isinstance(value, (bool, np.bool_)):\n        return float(default)\n    try:\n        parsed = float(value)\n    except (TypeError, ValueError, OverflowError):\n        return float(default)\n    return parsed if math.isfinite(parsed) else float(default)\n\n\ndef finite_int(value: Any, default: int = 0) -> int:\n    """Parse an integer-like metric defensively after JSON serialization."""\n    if value is None or isinstance(value, (bool, np.bool_)):\n        return int(default)\n    try:\n        parsed = float(value)\n    except (TypeError, ValueError, OverflowError):\n        return int(default)\n    if not math.isfinite(parsed):\n        return int(default)\n    return int(parsed)\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef canonical_hash(value: Any) -> str:\n    return hashlib.sha256(\n        json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")\n    ).hexdigest()\n\n\ndef json_public(value: Any) -> Any:\n    if isinstance(value, Mapping):\n        return {str(k): json_public(v) for k, v in value.items() if not str(k).startswith("_")}\n    if isinstance(value, (list, tuple)):\n        return [json_public(v) for v in value]\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    if isinstance(value, np.generic):\n        value = value.item()\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef atomic_json(path: Path, value: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(json_public(value), indent=2, sort_keys=True), encoding="utf-8")\n    os.replace(temp, path)\n\n\ndef atomic_csv(path: Path, frame: pd.DataFrame) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    frame.to_csv(temp, index=False)\n    os.replace(temp, path)\n\n\ndef env_tuple(name: str, default: Sequence[str]) -> Tuple[str, ...]:\n    raw = os.getenv(name, "").strip()\n    values = tuple(x.strip() for x in raw.split(",") if x.strip()) if raw else tuple(default)\n    if not values:\n        raise ValueError(f"{name} resolved to an empty tuple")\n    return values\n\n\ndef env_int_tuple(name: str, default: Sequence[int]) -> Tuple[int, ...]:\n    return tuple(int(x) for x in env_tuple(name, tuple(str(v) for v in default)))\n\n\ndef env_float_tuple(name: str, default: Sequence[float]) -> Tuple[float, ...]:\n    values = tuple(float(x) for x in env_tuple(name, tuple(str(v) for v in default)))\n    if not all(math.isfinite(x) for x in values):\n        raise ValueError(f"{name} contains non-finite values")\n    return values\n\n\ndef env_bool(name: str, default: bool) -> bool:\n    raw = os.getenv(name)\n    if raw is None:\n        return default\n    return raw.strip().lower() in {"1", "true", "yes", "on"}\n\n\n@dataclass(frozen=True)\nclass Config:\n    mode: str\n    seeds: Tuple[int, ...]\n    candidate_counts: Tuple[int, ...]\n    model_specs: Tuple[str, ...]\n    drive_mount: str\n    project_root_override: str\n    source_run_override: str\n    output_subdir: str\n    resume: bool\n    fail_on_hard_check: bool\n    expected_replay_count: int\n    expected_probe_count: int\n    random_seed: int\n    tree_estimators: int\n    random_forest_estimators: int\n    n_jobs: int\n    gate_threshold_grid: Tuple[float, ...]\n    expert_threshold_grid: Tuple[float, ...]\n    expert_margin_grid: Tuple[float, ...]\n    model_roundtrip_tolerance: float\n    replay_pooled_gain_min: float\n    replay_pooled_damage_max: float\n    replay_pooled_precision_min: float\n    replay_pooled_invocations_min: int\n    replay_pooled_rescues_min: int\n    replay_seed_gain_min: float\n    replay_seed_damage_max: float\n    replay_seed_precision_min: float\n    replay_seed_precision_min_invocations: int\n    probe_pooled_gain_min: float\n    probe_pooled_damage_max: float\n    probe_pooled_precision_min: float\n    probe_pooled_invocations_min: int\n    probe_pooled_rescues_min: int\n    strong_accuracy_target: float\n    exceptional_accuracy_target: float\n    original_target_accuracy: float\n\n    @classmethod\n    def from_env(cls) -> "Config":\n        mode = os.getenv("AKILI_V023T_MODE", "full").strip().lower()\n        if mode not in {"full", "smoke"}:\n            raise ValueError("AKILI_V023T_MODE must be full or smoke")\n        default_replay = (\n            400\n            if mode == "full"\n            else int(os.getenv("AKILI_V023T_SMOKE_REPLAY_COUNT", "72"))\n        )\n        default_probe = (\n            100\n            if mode == "full"\n            else int(os.getenv("AKILI_V023T_SMOKE_PROBE_COUNT", "36"))\n        )\n        default_trees = 220 if mode == "full" else 10\n        default_rf = 220 if mode == "full" else 10\n        cfg = cls(\n            mode=mode,\n            seeds=env_int_tuple("AKILI_V023T_SEEDS", (1, 2, 3)),\n            candidate_counts=env_int_tuple("AKILI_V023T_CANDIDATE_COUNTS", (2, 3)),\n            model_specs=env_tuple(\n                "AKILI_V023T_MODEL_SPECS",\n                tuple(MODEL_SPECS.keys()),\n            ),\n            drive_mount=os.getenv("AKILI_V023T_DRIVE_MOUNT", "/content/drive"),\n            project_root_override=os.getenv(\n                "AKILI_V023T_PROJECT_ROOT", ""\n            ).strip(),\n            source_run_override=os.getenv(\n                "AKILI_V023T_SOURCE_RUN_ROOT", ""\n            ).strip(),\n            output_subdir=os.getenv(\n                "AKILI_V023T_OUTPUT_SUBDIR",\n                "stage04/v0_23T_final_nonlinear_groupwise_verifier",\n            ),\n            resume=env_bool("AKILI_V023T_RESUME", True),\n            fail_on_hard_check=env_bool(\n                "AKILI_V023T_FAIL_ON_HARD_CHECK", True\n            ),\n            expected_replay_count=int(\n                os.getenv(\n                    "AKILI_V023T_EXPECTED_REPLAY_COUNT",\n                    str(default_replay),\n                )\n            ),\n            expected_probe_count=int(\n                os.getenv(\n                    "AKILI_V023T_EXPECTED_PROBE_COUNT",\n                    str(default_probe),\n                )\n            ),\n            random_seed=int(\n                os.getenv("AKILI_V023T_RANDOM_SEED", "23025")\n            ),\n            tree_estimators=int(\n                os.getenv(\n                    "AKILI_V023T_TREE_ESTIMATORS",\n                    str(default_trees),\n                )\n            ),\n            random_forest_estimators=int(\n                os.getenv(\n                    "AKILI_V023T_RANDOM_FOREST_ESTIMATORS",\n                    str(default_rf),\n                )\n            ),\n            n_jobs=int(\n                os.getenv(\n                    "AKILI_V023T_N_JOBS",\n                    "-1" if mode == "full" else "1",\n                )\n            ),\n            gate_threshold_grid=env_float_tuple(\n                "AKILI_V023T_GATE_THRESHOLD_GRID",\n                (\n                    0.10,\n                    0.15,\n                    0.20,\n                    0.25,\n                    0.30,\n                    0.35,\n                    0.40,\n                    0.45,\n                    0.50,\n                    0.55,\n                    0.60,\n                    0.65,\n                    0.70,\n                    0.75,\n                    0.80,\n                    0.85,\n                    0.90,\n                ),\n            ),\n            expert_threshold_grid=env_float_tuple(\n                "AKILI_V023T_EXPERT_THRESHOLD_GRID",\n                (\n                    0.10,\n                    0.20,\n                    0.30,\n                    0.40,\n                    0.50,\n                    0.60,\n                    0.70,\n                    0.80,\n                ),\n            ),\n            expert_margin_grid=env_float_tuple(\n                "AKILI_V023T_EXPERT_MARGIN_GRID",\n                (\n                    0.00,\n                    0.025,\n                    0.05,\n                    0.075,\n                    0.10,\n                    0.15,\n                    0.20,\n                    0.30,\n                ),\n            ),\n            model_roundtrip_tolerance=float(\n                os.getenv(\n                    "AKILI_V023T_MODEL_ROUNDTRIP_TOLERANCE",\n                    "1e-12",\n                )\n            ),\n            replay_pooled_gain_min=0.0,\n            replay_pooled_damage_max=0.015,\n            replay_pooled_precision_min=0.70,\n            replay_pooled_invocations_min=30,\n            replay_pooled_rescues_min=15,\n            replay_seed_gain_min=-0.01,\n            replay_seed_damage_max=0.03,\n            replay_seed_precision_min=0.25,\n            replay_seed_precision_min_invocations=4,\n            probe_pooled_gain_min=0.0,\n            probe_pooled_damage_max=0.015,\n            probe_pooled_precision_min=0.65,\n            probe_pooled_invocations_min=10,\n            probe_pooled_rescues_min=5,\n            strong_accuracy_target=0.72,\n            exceptional_accuracy_target=0.75,\n            original_target_accuracy=0.79,\n        )\n        cfg.validate()\n        return cfg\n\n    def validate(self) -> None:\n        if self.seeds != (1, 2, 3):\n            raise ValueError(\n                "v0.23T requires exactly seeds 1,2,3 for nested LOSO"\n            )\n        if set(self.candidate_counts) != {2, 3}:\n            raise ValueError(\n                "Source validation requires persisted top-2 and top-3 evidence"\n            )\n        unknown = set(self.model_specs) - set(MODEL_SPECS)\n        if unknown:\n            raise ValueError(\n                f"Unknown v0.23T model specs: {sorted(unknown)}"\n            )\n        if not self.model_specs:\n            raise ValueError("At least one model spec is required")\n        if self.expected_replay_count <= 0 or self.expected_probe_count <= 0:\n            raise ValueError("Expected evidence counts must be positive")\n        if self.tree_estimators <= 0 or self.random_forest_estimators <= 0:\n            raise ValueError("Nonlinear model iteration counts must be positive")\n        if self.n_jobs == 0:\n            raise ValueError("AKILI_V023T_N_JOBS cannot be zero")\n        if self.model_roundtrip_tolerance <= 0:\n            raise ValueError("Model roundtrip tolerance must be positive")\n        for name, grid in (\n            ("gate_threshold_grid", self.gate_threshold_grid),\n            ("expert_threshold_grid", self.expert_threshold_grid),\n            ("expert_margin_grid", self.expert_margin_grid),\n        ):\n            if not grid or not all(math.isfinite(x) for x in grid):\n                raise ValueError(f"{name} must be non-empty and finite")\n        if not all(0.0 <= x <= 1.0 for x in self.gate_threshold_grid):\n            raise ValueError("Gate thresholds must be in [0,1]")\n        if not all(0.0 <= x <= 1.0 for x in self.expert_threshold_grid):\n            raise ValueError("Expert thresholds must be in [0,1]")\n        if not all(0.0 <= x <= 1.0 for x in self.expert_margin_grid):\n            raise ValueError("Expert margins must be in [0,1]")\n        if not (\n            0.0\n            < self.strong_accuracy_target\n            <= self.exceptional_accuracy_target\n            <= self.original_target_accuracy\n            <= 1.0\n        ):\n            raise ValueError("Accuracy targets are inconsistent")\n\n    def public(self) -> Dict[str, Any]:\n        return json_public(asdict(self))\n\n\n@dataclass\nclass Evidence:\n    seed: int\n    split: str\n    candidate_count: int\n    indices: np.ndarray\n    labels: np.ndarray\n    action_predictions: np.ndarray\n    action_task_ids: np.ndarray\n    action_features: np.ndarray\n    action_correctness: np.ndarray\n    feature_names: Tuple[str, ...]\n    schema_hash: str\n    source_path: str = ""\n\n    @property\n    def n(self) -> int:\n        return int(self.labels.shape[0])\n\n    @property\n    def action_count(self) -> int:\n        return int(self.candidate_count + 1)\n\n    def validate(self) -> None:\n        n, a, f = self.n, self.action_count, len(self.feature_names)\n        if self.candidate_count not in {2, 3}:\n            raise ProtocolError("candidate_count must be 2 or 3")\n        if self.indices.shape != (n,) or self.labels.shape != (n,):\n            raise ProtocolError("index/label shape mismatch")\n        if self.action_predictions.shape != (n, a):\n            raise ProtocolError(f"prediction shape mismatch: {self.action_predictions.shape}")\n        if self.action_task_ids.shape != (n, a):\n            raise ProtocolError("task-id shape mismatch")\n        if self.action_features.shape != (n, a, f):\n            raise ProtocolError(f"feature shape mismatch: {self.action_features.shape}")\n        if self.action_correctness.shape != (n, a):\n            raise ProtocolError("correctness shape mismatch")\n        if f != EXPECTED_FEATURE_COUNT:\n            raise ProtocolError(f"Expected 26 features, found {f}")\n        if len(set(self.feature_names)) != f:\n            raise ProtocolError("Duplicate feature names")\n        if not np.isfinite(self.action_features).all():\n            raise ProtocolError("Non-finite action features")\n        expected = self.action_predictions == self.labels[:, None]\n        if not np.array_equal(expected, self.action_correctness.astype(bool)):\n            raise ProtocolError("Correctness tensor disagrees with predictions and labels")\n        if np.any(self.action_task_ids < 0):\n            raise ProtocolError("Negative task IDs")\n\n\ndef safe_torch_load(path: Path) -> Any:\n    try:\n        return torch.load(path, map_location="cpu", weights_only=True)\n    except TypeError:\n        return torch.load(path, map_location="cpu")\n\n\ndef to_numpy(value: Any, dtype: Optional[np.dtype] = None) -> np.ndarray:\n    if isinstance(value, torch.Tensor):\n        result = value.detach().cpu().numpy()\n    else:\n        result = np.asarray(value)\n    return result.astype(dtype, copy=False) if dtype is not None else result\n\n\ndef _canonical_array_bytes(value: Any, dtype: np.dtype) -> bytes:\n    """Return bytes using the exact dtype/contiguity contract used by v0.23R."""\n    array = to_numpy(value, dtype)\n    return np.ascontiguousarray(array, dtype=dtype).tobytes(order="C")\n\n\ndef evidence_digest(evidence: Evidence) -> str:\n    """Reproduce v0.23R ActionEvidence.evidence_digest exactly.\n\n    Integrity hashing is intentionally independent of the analysis dtype.\n    v0.23R serialized indices/labels/predictions/task IDs as int64,\n    action features as float32, and correctness as bool.  v0.23T may use\n    float64 copies for sklearn only after this source-compatible digest is\n    validated.\n    """\n    evidence.validate()\n    payload = {\n        "seed": int(evidence.seed),\n        "split": str(evidence.split),\n        "candidate_count": int(evidence.candidate_count),\n        "schema_hash": str(evidence.schema_hash),\n        "indices": hashlib.sha256(\n            _canonical_array_bytes(evidence.indices, np.int64)\n        ).hexdigest(),\n        "labels": hashlib.sha256(\n            _canonical_array_bytes(evidence.labels, np.int64)\n        ).hexdigest(),\n        "predictions": hashlib.sha256(\n            _canonical_array_bytes(evidence.action_predictions, np.int64)\n        ).hexdigest(),\n        "tasks": hashlib.sha256(\n            _canonical_array_bytes(evidence.action_task_ids, np.int64)\n        ).hexdigest(),\n        "features": hashlib.sha256(\n            _canonical_array_bytes(evidence.action_features, np.float32)\n        ).hexdigest(),\n        "correctness": hashlib.sha256(\n            _canonical_array_bytes(evidence.action_correctness, np.bool_)\n        ).hexdigest(),\n    }\n    return canonical_hash(payload)\n\n\ndef v023r_reference_digest_from_payload_item(\n    item: Mapping[str, Any],\n    schema_hash: str,\n) -> str:\n    """Independent reference implementation mirroring v0.23R from_payload()."""\n    payload = {\n        "seed": int(item["seed"]),\n        "split": str(item["split"]),\n        "candidate_count": int(item["candidate_count"]),\n        "schema_hash": str(schema_hash),\n        "indices": hashlib.sha256(\n            _canonical_array_bytes(item["indices"], np.int64)\n        ).hexdigest(),\n        "labels": hashlib.sha256(\n            _canonical_array_bytes(item["labels"], np.int64)\n        ).hexdigest(),\n        "predictions": hashlib.sha256(\n            _canonical_array_bytes(item["action_predictions"], np.int64)\n        ).hexdigest(),\n        "tasks": hashlib.sha256(\n            _canonical_array_bytes(item["action_task_ids"], np.int64)\n        ).hexdigest(),\n        "features": hashlib.sha256(\n            _canonical_array_bytes(item["action_features"], np.float32)\n        ).hexdigest(),\n        "correctness": hashlib.sha256(\n            _canonical_array_bytes(item["action_correctness"], np.bool_)\n        ).hexdigest(),\n    }\n    return canonical_hash(payload)\n\n\ndef _legacy_analysis_dtype_digest_for_regression_test(evidence: Evidence) -> str:\n    """The defective v2 behavior, retained only to prove the regression test."""\n    payload = {\n        "seed": evidence.seed,\n        "split": evidence.split,\n        "candidate_count": evidence.candidate_count,\n        "schema_hash": evidence.schema_hash,\n        "indices": hashlib.sha256(evidence.indices.tobytes()).hexdigest(),\n        "labels": hashlib.sha256(evidence.labels.tobytes()).hexdigest(),\n        "predictions": hashlib.sha256(evidence.action_predictions.tobytes()).hexdigest(),\n        "tasks": hashlib.sha256(evidence.action_task_ids.tobytes()).hexdigest(),\n        "features": hashlib.sha256(evidence.action_features.tobytes()).hexdigest(),\n        "correctness": hashlib.sha256(evidence.action_correctness.tobytes()).hexdigest(),\n    }\n    return canonical_hash(payload)\n\n\ndef load_evidence_file(path: Path) -> Tuple[Dict[int, Evidence], Mapping[str, Any], Mapping[str, Any]]:\n    payload = safe_torch_load(path)\n    if not isinstance(payload, Mapping):\n        raise ProtocolError(f"Evidence artifact is not a mapping: {path}")\n    if payload.get("protocol") != SOURCE_PROTOCOL:\n        raise ProtocolError(f"Unexpected source protocol in {path}: {payload.get(\'protocol\')}")\n    if payload.get("schema_version") != EXPECTED_SCHEMA_VERSION:\n        raise ProtocolError(f"Unexpected schema version in {path}")\n    schema_hash = str(payload.get("schema_hash", ""))\n    names = tuple(str(x) for x in payload.get("feature_names", ()))\n    if len(names) != EXPECTED_FEATURE_COUNT:\n        raise ProtocolError(f"Expected 26 feature names in {path}")\n    result: Dict[int, Evidence] = {}\n    for key, item in payload.get("candidate_evidence", {}).items():\n        k = int(key)\n        item_names = tuple(str(x) for x in item.get("feature_names", names))\n        evidence = Evidence(\n            seed=int(item["seed"]),\n            split=str(item["split"]),\n            candidate_count=k,\n            indices=to_numpy(item["indices"], np.int64),\n            labels=to_numpy(item["labels"], np.int64),\n            action_predictions=to_numpy(item["action_predictions"], np.int64),\n            action_task_ids=to_numpy(item["action_task_ids"], np.int64),\n            action_features=to_numpy(item["action_features"], np.float64),\n            action_correctness=to_numpy(item["action_correctness"], bool),\n            feature_names=item_names,\n            schema_hash=str(item.get("schema_hash", schema_hash)),\n            source_path=str(path),\n        )\n        evidence.validate()\n        if evidence.schema_hash != schema_hash or evidence.feature_names != names:\n            raise ProtocolError(f"Nested evidence schema mismatch in {path}")\n        result[k] = evidence\n    if set(result) != {2, 3}:\n        raise ProtocolError(f"Evidence artifact must contain top-2 and top-3: {path}")\n    return result, payload.get("metadata", {}), payload\n\n\ndef _project_score(path: Path) -> Tuple[int, int, str]:\n    markers = sum((path / name).is_dir() for name in ("stage03", "stage04", "data"))\n    name = int(path.name.lower() == "akm_clr")\n    return markers, name, str(path)\n\n\ndef resolve_project_root(cfg: Config) -> Path:\n    """Resolve AKM_CLR without recursive traversal through Drive shortcuts.\n\n    Google Drive\'s Colab mount does not reliably support Path.rglob across\n    `.shortcut-targets-by-id`.  Traverse the small, known directory depth\n    explicitly, matching the discovery strategy already validated by v0.23R.\n    """\n    if cfg.project_root_override:\n        candidate = Path(cfg.project_root_override).expanduser()\n        if not candidate.is_dir():\n            raise FileNotFoundError(f"Configured project root does not exist: {candidate}")\n        resolved = candidate.resolve()\n        if _project_score(resolved)[0] < 2:\n            raise FileNotFoundError(f"Configured project root lacks expected markers: {resolved}")\n        print(f"[root] {resolved}", flush=True)\n        return resolved\n\n    mount = Path(cfg.drive_mount)\n    mydrive = mount / "MyDrive"\n    shortcuts = mount / ".shortcut-targets-by-id"\n    candidates: Dict[str, Path] = {}\n    inspected: Dict[str, List[str]] = {}\n\n    def safe_children(path: Path, limit: int) -> List[Path]:\n        if not path.is_dir():\n            return []\n        try:\n            children = sorted(path.iterdir(), key=lambda p: p.name.lower())[:limit]\n            inspected[str(path)] = [child.name for child in children[:50]]\n            return children\n        except OSError as exc:\n            inspected[str(path)] = [f"<unreadable: {type(exc).__name__}: {exc}>"]\n            return []\n\n    def consider(path: Path) -> None:\n        try:\n            resolved = path.resolve()\n        except OSError:\n            return\n        if not resolved.is_dir():\n            return\n        score, name_bonus, _ = _project_score(resolved)\n        if score >= 2 or (name_bonus and score >= 1):\n            candidates[str(resolved)] = resolved\n\n    # Common direct MyDrive placements and at most two nested folder levels.\n    if mydrive.is_dir():\n        consider(mydrive / "AKM_CLR")\n        for child in safe_children(mydrive, 500):\n            if not child.is_dir():\n                continue\n            consider(child)\n            for grandchild in safe_children(child, 500):\n                if grandchild.is_dir():\n                    consider(grandchild)\n\n    # Shortcut layout used by the real project:\n    # .shortcut-targets-by-id/<shortcut-id>/ALL/AKM_CLR\n    if shortcuts.is_dir():\n        for shortcut_id in safe_children(shortcuts, 200):\n            if not shortcut_id.is_dir():\n                continue\n            consider(shortcut_id)\n            for level1 in safe_children(shortcut_id, 200):\n                if not level1.is_dir():\n                    continue\n                consider(level1)\n                for level2 in safe_children(level1, 500):\n                    if level2.is_dir():\n                        consider(level2)\n\n    if not candidates:\n        diagnostics = {\n            "drive_mount": str(mount),\n            "mydrive_exists": mydrive.is_dir(),\n            "shortcuts_exists": shortcuts.is_dir(),\n            "inspected": inspected,\n        }\n        raise FileNotFoundError(\n            "Could not discover AKM_CLR using shortcut-aware bounded traversal. "\n            "Set AKILI_V023T_PROJECT_ROOT only if the project was moved outside the usual Drive tree.\\n"\n            + json.dumps(diagnostics, indent=2)\n        )\n\n    ranked = sorted(candidates.values(), key=_project_score, reverse=True)\n    best = ranked[0]\n    best_key = _project_score(best)[:2]\n    tied = [path for path in ranked if _project_score(path)[:2] == best_key]\n    if len(tied) > 1:\n        named = [path for path in tied if path.name.lower() == "akm_clr"]\n        if len(named) == 1:\n            best = named[0]\n        else:\n            raise ProtocolError(\n                "Multiple equally plausible project roots found. Set AKILI_V023T_PROJECT_ROOT: "\n                + ", ".join(map(str, tied[:20]))\n            )\n\n    print(f"[root] {best}", flush=True)\n    return best\n\n\ndef source_run_complete(path: Path, cfg: Config) -> bool:\n    if not path.is_dir():\n        return False\n    if not (path / "completion.json").is_file() or not (path / "hard_checks.json").is_file():\n        return False\n    try:\n        completion = json.loads((path / "completion.json").read_text(encoding="utf-8"))\n        hard = json.loads((path / "hard_checks.json").read_text(encoding="utf-8"))\n    except Exception:\n        return False\n    if completion.get("protocol") != SOURCE_PROTOCOL or not bool(hard.get("all_passed")):\n        return False\n    for seed in cfg.seeds:\n        seed_root = path / f"seed_{seed}"\n        required = (\n            seed_root / "action_feature_manifest.json",\n            seed_root / "replay_action_evidence.pt",\n            seed_root / "protected_probe_action_evidence.pt",\n            seed_root / "evidence_hashes.json",\n        )\n        if not all(p.is_file() for p in required):\n            return False\n    return True\n\n\ndef resolve_source_run(project_root: Path, cfg: Config) -> Path:\n    if cfg.source_run_override:\n        path = Path(cfg.source_run_override).expanduser().resolve()\n        if not source_run_complete(path, cfg):\n            raise FileNotFoundError(f"Configured v0.23R source run is incomplete: {path}")\n        return path\n    stage = project_root / "stage04" / "v0_23R_reconstructed_action_evidence"\n    candidates = [p.resolve() for p in stage.glob("run_*") if source_run_complete(p, cfg)] if stage.is_dir() else []\n    if not candidates:\n        raise FileNotFoundError(\n            "No complete v0.23R persisted-evidence run found. Set AKILI_V023T_SOURCE_RUN_ROOT."\n        )\n    candidates.sort(key=lambda p: ((p / "completion.json").stat().st_mtime_ns, str(p)), reverse=True)\n    return candidates[0]\n\n\ndef source_files(source_run: Path, cfg: Config) -> List[Path]:\n    paths = [source_run / "completion.json", source_run / "hard_checks.json", source_run / "aggregate_summary.json"]\n    for seed in cfg.seeds:\n        root = source_run / f"seed_{seed}"\n        paths.extend(\n            [\n                root / "action_feature_manifest.json",\n                root / "replay_action_evidence.pt",\n                root / "protected_probe_action_evidence.pt",\n                root / "evidence_hashes.json",\n            ]\n        )\n    return sorted({p.resolve() for p in paths if p.is_file()})\n\n\ndef load_replay_and_probe_catalog(\n    source_run: Path,\n    cfg: Config,\n) -> Tuple[Dict[int, Dict[int, Evidence]], Dict[int, Dict[str, Any]], Dict[str, Any]]:\n    replay: Dict[int, Dict[int, Evidence]] = {}\n    probe_catalog: Dict[int, Dict[str, Any]] = {}\n    schema_hashes = set()\n    feature_names: Optional[Tuple[str, ...]] = None\n    source_hard_checks = json.loads((source_run / "hard_checks.json").read_text(encoding="utf-8"))\n    if not bool(source_hard_checks.get("all_passed")):\n        raise ProtocolError("The source v0.23R run did not pass its hard checks")\n    validation: Dict[str, Any] = {\n        "seeds": {},\n        "source_v023r_hard_checks_passed": True,\n        "protected_probe_deserialized_during_initial_load": False,\n        "source_digest_contract": {\n            "indices": "int64",\n            "labels": "int64",\n            "action_predictions": "int64",\n            "action_task_ids": "int64",\n            "action_features": "float32",\n            "action_correctness": "bool",\n            "canonical_json": "sort_keys=True,separators=(comma,colon)",\n        },\n        "analysis_feature_dtype": "float64_after_integrity_validation",\n    }\n    for seed in cfg.seeds:\n        seed_root = source_run / f"seed_{seed}"\n        manifest = json.loads((seed_root / "action_feature_manifest.json").read_text(encoding="utf-8"))\n        hashes = json.loads((seed_root / "evidence_hashes.json").read_text(encoding="utf-8"))\n        replay_path = seed_root / "replay_action_evidence.pt"\n        probe_path = seed_root / "protected_probe_action_evidence.pt"\n        if sha256_file(replay_path) != hashes["replay"]["sha256"]:\n            raise ProtocolError(f"Replay file hash mismatch for seed {seed}")\n        if sha256_file(probe_path) != hashes["probe"]["sha256"]:\n            raise ProtocolError(f"Probe file hash mismatch for seed {seed}")\n        replay[seed], replay_meta, _ = load_evidence_file(replay_path)\n        if int(replay_meta.get("seed", seed)) != seed:\n            raise ProtocolError(f"Metadata seed mismatch for seed {seed}")\n        if manifest.get("metadata") != replay_meta:\n            raise ProtocolError(f"Manifest metadata mismatch for seed {seed}")\n        for k in cfg.candidate_counts:\n            r = replay[seed][k]\n            if r.seed != seed:\n                raise ProtocolError("Replay evidence seed mismatch")\n            if r.n != cfg.expected_replay_count:\n                raise ProtocolError(\n                    f"Replay count mismatch seed={seed} top{k}: {r.n} != {cfg.expected_replay_count}"\n                )\n            if not np.array_equal(r.labels, replay[seed][3].labels):\n                raise ProtocolError(f"Candidate-count replay labels differ seed={seed}")\n            if not np.array_equal(r.action_predictions[:, 0], replay[seed][3].action_predictions[:, 0]):\n                raise ProtocolError(f"KEEP predictions differ by candidate count seed={seed}")\n            schema_hashes.add(r.schema_hash)\n            if feature_names is None:\n                feature_names = r.feature_names\n            if r.feature_names != feature_names:\n                raise ProtocolError("Feature names differ across replay seeds")\n            expected_digest_r = hashes["replay"]["evidence_digests"][str(k)]\n            if evidence_digest(r) != expected_digest_r:\n                raise ProtocolError(f"Replay evidence digest mismatch seed={seed} top{k}")\n            if int(hashes["probe"]["counts"][str(k)]) != cfg.expected_probe_count:\n                raise ProtocolError(\n                    f"Probe catalog count mismatch seed={seed} top{k}: "\n                    f"{hashes[\'probe\'][\'counts\'][str(k)]} != {cfg.expected_probe_count}"\n                )\n        probe_catalog[seed] = {\n            "path": str(probe_path),\n            "metadata": replay_meta,\n            "sha256": hashes["probe"]["sha256"],\n            "evidence_digests": hashes["probe"]["evidence_digests"],\n            "counts": hashes["probe"]["counts"],\n        }\n        validation["seeds"][str(seed)] = {\n            "replay_sha256": hashes["replay"]["sha256"],\n            "probe_sha256_catalog_only": hashes["probe"]["sha256"],\n            "metadata": replay_meta,\n        }\n    if len(schema_hashes) != 1 or feature_names is None:\n        raise ProtocolError("Replay evidence schema is not shared across all seeds")\n    validation["schema_hash"] = next(iter(schema_hashes))\n    validation["feature_names"] = list(feature_names)\n    validation["feature_count"] = len(feature_names)\n    return replay, probe_catalog, validation\n\n\ndef load_protected_probe_after_replay_selection(\n    probe_catalog: Mapping[int, Mapping[str, Any]],\n    replay: Mapping[int, Mapping[int, Evidence]],\n    cfg: Config,\n) -> Tuple[Dict[int, Dict[int, Evidence]], Dict[str, Any]]:\n    probe: Dict[int, Dict[int, Evidence]] = {}\n    audit: Dict[str, Any] = {"deserialized_seed_files": 0, "seeds": {}}\n    for seed in cfg.seeds:\n        entry = probe_catalog[seed]\n        path = Path(str(entry["path"]))\n        if sha256_file(path) != str(entry["sha256"]):\n            raise ProtocolError(f"Protected-probe file hash changed before access seed={seed}")\n        loaded, metadata, _ = load_evidence_file(path)\n        audit["deserialized_seed_files"] += 1\n        if metadata != entry["metadata"]:\n            raise ProtocolError(f"Protected-probe metadata mismatch seed={seed}")\n        for k in cfg.candidate_counts:\n            evidence = loaded[k]\n            if evidence.n != cfg.expected_probe_count:\n                raise ProtocolError(f"Protected-probe count mismatch seed={seed} top{k}")\n            if evidence_digest(evidence) != entry["evidence_digests"][str(k)]:\n                raise ProtocolError(f"Protected-probe digest mismatch seed={seed} top{k}")\n            if set(evidence.indices.tolist()) & set(replay[seed][k].indices.tolist()):\n                raise ProtocolError(f"Replay/protected-probe overlap seed={seed} top{k}")\n            if evidence.feature_names != replay[seed][k].feature_names:\n                raise ProtocolError(f"Protected-probe feature schema mismatch seed={seed} top{k}")\n        probe[seed] = loaded\n        audit["seeds"][str(seed)] = {\n            "path": str(path),\n            "sha256": entry["sha256"],\n            "top2_count": loaded[2].n,\n            "top3_count": loaded[3].n,\n        }\n    audit["replay_probe_disjoint"] = True\n    return probe, audit\n\n\ndef combine(parts: Sequence[Evidence], split: str) -> Evidence:\n    if not parts:\n        raise ValueError("No evidence parts")\n    k = parts[0].candidate_count\n    names = parts[0].feature_names\n    schema = parts[0].schema_hash\n    if any(p.candidate_count != k or p.feature_names != names or p.schema_hash != schema for p in parts):\n        raise ProtocolError("Cannot combine incompatible evidence")\n    result = Evidence(\n        seed=-1,\n        split=split,\n        candidate_count=k,\n        indices=np.arange(sum(p.n for p in parts), dtype=np.int64),\n        labels=np.concatenate([p.labels for p in parts]),\n        action_predictions=np.concatenate([p.action_predictions for p in parts], axis=0),\n        action_task_ids=np.concatenate([p.action_task_ids for p in parts], axis=0),\n        action_features=np.concatenate([p.action_features for p in parts], axis=0),\n        action_correctness=np.concatenate([p.action_correctness for p in parts], axis=0),\n        feature_names=names,\n        schema_hash=schema,\n    )\n    result.validate()\n    return result\n\n\ndef capacity_metrics(evidence: Evidence) -> Dict[str, Any]:\n    evidence.validate()\n    correct = evidence.action_correctness.astype(bool)\n    keep = correct[:, 0]\n    expert = correct[:, 1:]\n    union = np.any(correct, axis=1)\n    expert_union = np.any(expert, axis=1)\n    counts = correct.sum(axis=1)\n    metrics: Dict[str, Any] = {\n        "examples": evidence.n,\n        "baseline_accuracy": float(keep.mean()),\n        "available_action_union_accuracy": float(union.mean()),\n        "expert_only_union_accuracy": float(expert_union.mean()),\n        "rescue_opportunity_count": int(((~keep) & expert_union).sum()),\n        "rescue_opportunity_rate": float(((~keep) & expert_union).mean()),\n        "no_available_correct_action_count": int((~union).sum()),\n        "no_available_correct_action_rate": float((~union).mean()),\n        "multiple_correct_actions_count": int((counts > 1).sum()),\n        "multiple_correct_actions_rate": float((counts > 1).mean()),\n        "only_keep_correct_count": int((keep & (expert.sum(axis=1) == 0)).sum()),\n        "keep_wrong_no_expert_correct_count": int(((~keep) & (~expert_union)).sum()),\n    }\n    for rank in range(1, evidence.action_count):\n        only = correct[:, rank] & (correct.sum(axis=1) == 1)\n        metrics[f"only_expert_{rank}_correct_count"] = int(only.sum())\n        metrics[f"expert_{rank}_correct_rate"] = float(correct[:, rank].mean())\n    return metrics\n\n\ndef run_capacity_audit(replay: Mapping[int, Mapping[int, Evidence]], output_root: Path) -> Dict[str, Any]:\n    rows: List[Dict[str, Any]] = []\n    per_example_rows: List[Dict[str, Any]] = []\n    for seed in sorted(replay):\n        top3 = replay[seed][3]\n        correct = top3.action_correctness.astype(bool)\n        for i in range(top3.n):\n            row = {\n                "seed": seed,\n                "example_index": int(top3.indices[i]),\n                "label": int(top3.labels[i]),\n                "keep_prediction": int(top3.action_predictions[i, 0]),\n                "keep_correct": bool(correct[i, 0]),\n                "expert1_correct": bool(correct[i, 1]),\n                "expert2_correct": bool(correct[i, 2]),\n                "expert3_correct": bool(correct[i, 3]),\n                "top1_union_correct": bool(correct[i, :2].any()),\n                "top2_union_correct": bool(correct[i, :3].any()),\n                "top3_union_correct": bool(correct[i, :4].any()),\n                "rescue_available_top3": bool((not correct[i, 0]) and correct[i, 1:].any()),\n                "no_action_correct": bool(not correct[i, :4].any()),\n                "correct_action_count": int(correct[i, :4].sum()),\n            }\n            per_example_rows.append(row)\n        base = float(correct[:, 0].mean())\n        union1 = float(correct[:, :2].any(axis=1).mean())\n        union2 = float(correct[:, :3].any(axis=1).mean())\n        union3 = float(correct[:, :4].any(axis=1).mean())\n        row = {\n            "scope": f"seed_{seed}",\n            **capacity_metrics(top3),\n            "keep_plus_expert1_accuracy": union1,\n            "top2_action_union_accuracy": union2,\n            "top3_action_union_accuracy": union3,\n            "top1_increment_over_keep": union1 - base,\n            "expert2_increment_over_top1": union2 - union1,\n            "expert3_increment_over_top2": union3 - union2,\n            "gap_from_top3_union_to_79pct": 0.79 - union3,\n        }\n        rows.append(row)\n    pooled = combine([replay[s][3] for s in sorted(replay)], "pooled_replay_top3")\n    correct = pooled.action_correctness.astype(bool)\n    base = float(correct[:, 0].mean())\n    union1 = float(correct[:, :2].any(axis=1).mean())\n    union2 = float(correct[:, :3].any(axis=1).mean())\n    union3 = float(correct[:, :4].any(axis=1).mean())\n    pooled_row = {\n        "scope": "pooled",\n        **capacity_metrics(pooled),\n        "keep_plus_expert1_accuracy": union1,\n        "top2_action_union_accuracy": union2,\n        "top3_action_union_accuracy": union3,\n        "top1_increment_over_keep": union1 - base,\n        "expert2_increment_over_top1": union2 - union1,\n        "expert3_increment_over_top2": union3 - union2,\n        "gap_from_top3_union_to_79pct": 0.79 - union3,\n    }\n    rows.append(pooled_row)\n    frame = pd.DataFrame(rows)\n    atomic_csv(output_root / "capacity_summary.csv", frame)\n    atomic_csv(output_root / "capacity_per_example_top3.csv", pd.DataFrame(per_example_rows))\n    summary = {\n        "pooled": pooled_row,\n        "per_seed": {row["scope"]: row for row in rows if row["scope"] != "pooled"},\n        "interpretation": (\n            "top3_action_union_accuracy is the maximum achievable replay accuracy using only KEEP and the three persisted candidate experts."\n        ),\n    }\n    atomic_json(output_root / "capacity_summary.json", summary)\n    return summary\n\n\ndef safe_auc(y: np.ndarray, score: np.ndarray) -> float:\n    y = np.asarray(y, dtype=np.int64)\n    score = np.asarray(score, dtype=np.float64)\n    if np.unique(y).size < 2 or not np.isfinite(score).all():\n        return float("nan")\n    return float(roc_auc_score(y, score))\n\n\ndef class_balanced_weights(labels: np.ndarray) -> np.ndarray:\n    y = np.asarray(labels)\n    result = np.ones(y.shape[0], dtype=np.float64)\n    classes, counts = np.unique(y, return_counts=True)\n    if classes.size <= 1:\n        return result\n    n = float(y.shape[0])\n    for cls, count in zip(classes, counts):\n        result[y == cls] = n / (float(classes.size) * float(count))\n    return result\n\n\ndef normalized_vote_count(values: np.ndarray) -> float:\n    _, counts = np.unique(values, return_counts=True)\n    return float(counts.max() / max(values.size, 1))\n\n\ndef relation_features(data: Evidence) -> Tuple[np.ndarray, Tuple[str, ...]]:\n    if data.candidate_count != CANDIDATE_COUNT:\n        raise ProtocolError("v0.23T relation features require top-3 evidence")\n    predictions = data.action_predictions\n    tasks = data.action_task_ids\n    n = data.n\n    rows = np.zeros((n, 18), dtype=np.float32)\n    for i in range(n):\n        expert_predictions = predictions[i, 1:]\n        expert_tasks = tasks[i, 1:]\n        rows[i, 0] = np.unique(predictions[i]).size / 4.0\n        rows[i, 1] = np.unique(expert_predictions).size / 3.0\n        rows[i, 2] = np.mean(expert_predictions == predictions[i, 0])\n        rows[i, 3] = normalized_vote_count(expert_predictions)\n        rows[i, 4] = float(np.unique(expert_predictions).size == 1)\n        rows[i, 5] = float(np.unique(expert_predictions).size < 3)\n        rows[i, 6] = np.unique(tasks[i]).size / 4.0\n        rows[i, 7] = np.unique(expert_tasks).size / 3.0\n        rows[i, 8] = np.mean(expert_tasks == tasks[i, 0])\n        rows[i, 9] = normalized_vote_count(expert_tasks)\n        rows[i, 10] = float(np.unique(expert_tasks).size == 1)\n        rows[i, 11] = float(np.unique(expert_tasks).size < 3)\n        rows[i, 12] = float(expert_predictions[0] == expert_predictions[1])\n        rows[i, 13] = float(expert_predictions[0] == expert_predictions[2])\n        rows[i, 14] = float(expert_predictions[1] == expert_predictions[2])\n        rows[i, 15] = float(expert_tasks[0] == expert_tasks[1])\n        rows[i, 16] = float(expert_tasks[0] == expert_tasks[2])\n        rows[i, 17] = float(expert_tasks[1] == expert_tasks[2])\n    names = (\n        "relation::unique_action_prediction_fraction",\n        "relation::unique_expert_prediction_fraction",\n        "relation::experts_matching_keep_prediction_fraction",\n        "relation::max_expert_prediction_vote_fraction",\n        "relation::all_experts_same_prediction",\n        "relation::any_expert_pair_same_prediction",\n        "relation::unique_action_task_fraction",\n        "relation::unique_expert_task_fraction",\n        "relation::experts_matching_keep_task_fraction",\n        "relation::max_expert_task_vote_fraction",\n        "relation::all_experts_same_task",\n        "relation::any_expert_pair_same_task",\n        "relation::expert1_prediction_equals_expert2",\n        "relation::expert1_prediction_equals_expert3",\n        "relation::expert2_prediction_equals_expert3",\n        "relation::expert1_task_equals_expert2",\n        "relation::expert1_task_equals_expert3",\n        "relation::expert2_task_equals_expert3",\n    )\n    return rows, names\n\n\ndef build_group_features(\n    data: Evidence,\n) -> Tuple[np.ndarray, Tuple[str, ...]]:\n    data.validate()\n    if data.candidate_count != CANDIDATE_COUNT:\n        raise ProtocolError("v0.23T uses top-3 evidence only")\n    action = data.action_features.astype(np.float32, copy=False)\n    keep = action[:, 0, :]\n    experts = action[:, 1:, :]\n    mean = experts.mean(axis=1)\n    std = experts.std(axis=1)\n    minimum = experts.min(axis=1)\n    maximum = experts.max(axis=1)\n    relation, relation_names = relation_features(data)\n\n    parts = [\n        keep,\n        experts.reshape(data.n, -1),\n        (experts - keep[:, None, :]).reshape(data.n, -1),\n        mean,\n        std,\n        minimum,\n        maximum,\n        mean - keep,\n        maximum - keep,\n        minimum - keep,\n        relation,\n    ]\n    names: List[str] = []\n    names.extend(f"keep::{name}" for name in data.feature_names)\n    for rank in range(1, 4):\n        names.extend(\n            f"expert{rank}::{name}" for name in data.feature_names\n        )\n    for rank in range(1, 4):\n        names.extend(\n            f"expert{rank}_minus_keep::{name}"\n            for name in data.feature_names\n        )\n    for aggregate in ("mean", "std", "min", "max"):\n        names.extend(\n            f"expert_{aggregate}::{name}" for name in data.feature_names\n        )\n    for aggregate in ("mean", "max", "min"):\n        names.extend(\n            f"expert_{aggregate}_minus_keep::{name}"\n            for name in data.feature_names\n        )\n    names.extend(relation_names)\n\n    result = np.concatenate(parts, axis=1).astype(np.float32, copy=False)\n    if result.shape != (data.n, len(names)):\n        raise ProtocolError(\n            f"Group feature shape mismatch: {result.shape} vs "\n            f"{(data.n, len(names))}"\n        )\n    if not np.isfinite(result).all():\n        raise ProtocolError("Non-finite group features")\n    return result, tuple(names)\n\n\ndef build_ranker_features(\n    data: Evidence,\n    group_features: Optional[np.ndarray] = None,\n    group_names: Optional[Tuple[str, ...]] = None,\n) -> Tuple[np.ndarray, Tuple[str, ...]]:\n    if data.candidate_count != CANDIDATE_COUNT:\n        raise ProtocolError("v0.23T ranker requires top-3 evidence")\n    if group_features is None or group_names is None:\n        group_features, group_names = build_group_features(data)\n    action = data.action_features.astype(np.float32, copy=False)\n    keep = action[:, 0, :]\n    experts = action[:, 1:, :]\n    rows: List[np.ndarray] = []\n\n    for rank in range(3):\n        current = experts[:, rank, :]\n        other_indices = [index for index in range(3) if index != rank]\n        others = experts[:, other_indices, :]\n        other_mean = others.mean(axis=1)\n        other_max = others.max(axis=1)\n        other_min = others.min(axis=1)\n        other_std = others.std(axis=1)\n\n        prediction_match_fraction = np.mean(\n            data.action_predictions[:, other_indices + np.ones(\n                len(other_indices), dtype=np.int64\n            )]\n            == data.action_predictions[:, [rank + 1]],\n            axis=1,\n        ).reshape(-1, 1)\n        task_match_fraction = np.mean(\n            data.action_task_ids[:, other_indices + np.ones(\n                len(other_indices), dtype=np.int64\n            )]\n            == data.action_task_ids[:, [rank + 1]],\n            axis=1,\n        ).reshape(-1, 1)\n        relations = np.concatenate(\n            [\n                (\n                    data.action_predictions[:, rank + 1]\n                    == data.action_predictions[:, 0]\n                ).astype(np.float32).reshape(-1, 1),\n                (\n                    data.action_task_ids[:, rank + 1]\n                    == data.action_task_ids[:, 0]\n                ).astype(np.float32).reshape(-1, 1),\n                prediction_match_fraction.astype(np.float32),\n                task_match_fraction.astype(np.float32),\n                np.full(\n                    (data.n, 1),\n                    float(rank + 1) / 3.0,\n                    dtype=np.float32,\n                ),\n            ],\n            axis=1,\n        )\n        specific = np.concatenate(\n            [\n                current,\n                current - keep,\n                current - other_mean,\n                current - other_max,\n                current - other_min,\n                other_std,\n                relations,\n            ],\n            axis=1,\n        )\n        rows.append(\n            np.concatenate([group_features, specific], axis=1)\n        )\n\n    result = np.stack(rows, axis=1).astype(np.float32, copy=False)\n    specific_names: List[str] = []\n    specific_names.extend(\n        f"current_expert::{name}" for name in data.feature_names\n    )\n    specific_names.extend(\n        f"current_expert_minus_keep::{name}"\n        for name in data.feature_names\n    )\n    specific_names.extend(\n        f"current_expert_minus_other_mean::{name}"\n        for name in data.feature_names\n    )\n    specific_names.extend(\n        f"current_expert_minus_other_max::{name}"\n        for name in data.feature_names\n    )\n    specific_names.extend(\n        f"current_expert_minus_other_min::{name}"\n        for name in data.feature_names\n    )\n    specific_names.extend(\n        f"other_expert_std::{name}" for name in data.feature_names\n    )\n    specific_names.extend(\n        (\n            "current_expert::prediction_matches_keep",\n            "current_expert::task_matches_keep",\n            "current_expert::other_prediction_match_fraction",\n            "current_expert::other_task_match_fraction",\n            "current_expert::rank_normalized",\n        )\n    )\n    names = tuple(group_names) + tuple(specific_names)\n    if result.shape != (data.n, 3, len(names)):\n        raise ProtocolError(\n            f"Ranker feature shape mismatch: {result.shape}"\n        )\n    if not np.isfinite(result).all():\n        raise ProtocolError("Non-finite ranker features")\n    return result, names\n\n\ndef rescue_target(data: Evidence) -> np.ndarray:\n    correct = data.action_correctness.astype(bool)\n    return ((~correct[:, 0]) & correct[:, 1:].any(axis=1)).astype(\n        np.int64\n    )\n\n\ndef deterministic_joint_target(data: Evidence) -> np.ndarray:\n    correct = data.action_correctness.astype(bool)\n    target = np.zeros(data.n, dtype=np.int64)\n    opportunities = (~correct[:, 0]) & correct[:, 1:].any(axis=1)\n    for action in range(1, 4):\n        choose = opportunities & (target == 0) & correct[:, action]\n        target[choose] = action\n    return target\n\n\ndef ranker_targets_and_weights(\n    data: Evidence,\n) -> Tuple[np.ndarray, np.ndarray]:\n    correct = data.action_correctness[:, 1:].astype(np.int64)\n    labels = correct.reshape(-1)\n    balanced = class_balanced_weights(labels).reshape(data.n, 3)\n    keep_correct = data.action_correctness[:, 0].astype(bool)\n    expert_correct = data.action_correctness[:, 1:].astype(bool)\n    differs = (\n        data.action_predictions[:, 1:]\n        != data.action_predictions[:, [0]]\n    )\n    multiplier = np.ones((data.n, 3), dtype=np.float64)\n    multiplier[\n        (~keep_correct)[:, None] & expert_correct & differs\n    ] = 2.0\n    multiplier[\n        keep_correct[:, None] & (~expert_correct) & differs\n    ] = 3.0\n    multiplier[~differs] = 0.5\n    weights = balanced * multiplier\n    return labels, weights.reshape(-1)\n\n\n@dataclass\nclass BinaryProbabilityBundle:\n    backend_names: Tuple[str, ...]\n    backend_weights: Tuple[float, ...]\n    models: Tuple[Any, ...]\n    constant_probability: Optional[float] = None\n\n    def predict_positive(self, features: np.ndarray) -> np.ndarray:\n        x = np.asarray(features, dtype=np.float32)\n        if self.constant_probability is not None:\n            result = np.full(\n                x.shape[0],\n                float(self.constant_probability),\n                dtype=np.float64,\n            )\n        else:\n            result = np.zeros(x.shape[0], dtype=np.float64)\n            for weight, model in zip(\n                self.backend_weights, self.models\n            ):\n                probabilities = model.predict_proba(x)\n                classes = np.asarray(model.classes_, dtype=np.int64)\n                positive_columns = np.where(classes == 1)[0]\n                component = (\n                    probabilities[:, int(positive_columns[0])]\n                    if positive_columns.size\n                    else np.zeros(x.shape[0], dtype=np.float64)\n                )\n                result += float(weight) * component\n        if result.shape != (x.shape[0],):\n            raise ProtocolError("Binary probability shape mismatch")\n        if not np.isfinite(result).all():\n            raise ProtocolError("Non-finite binary probabilities")\n        return np.clip(result, 0.0, 1.0)\n\n\n@dataclass\nclass MultiClassProbabilityBundle:\n    class_count: int\n    backend_names: Tuple[str, ...]\n    backend_weights: Tuple[float, ...]\n    models: Tuple[Any, ...]\n    constant_class: Optional[int] = None\n\n    def predict_full(self, features: np.ndarray) -> np.ndarray:\n        x = np.asarray(features, dtype=np.float32)\n        result = np.zeros(\n            (x.shape[0], self.class_count), dtype=np.float64\n        )\n        if self.constant_class is not None:\n            result[:, int(self.constant_class)] = 1.0\n        else:\n            for weight, model in zip(\n                self.backend_weights, self.models\n            ):\n                probabilities = model.predict_proba(x)\n                classes = np.asarray(model.classes_, dtype=np.int64)\n                for column, cls in enumerate(classes):\n                    if 0 <= int(cls) < self.class_count:\n                        result[:, int(cls)] += (\n                            float(weight) * probabilities[:, column]\n                        )\n        row_sums = result.sum(axis=1, keepdims=True)\n        zero = row_sums[:, 0] <= 0.0\n        if np.any(zero):\n            result[zero, 0] = 1.0\n            row_sums = result.sum(axis=1, keepdims=True)\n        result = result / row_sums\n        if not np.isfinite(result).all():\n            raise ProtocolError("Non-finite multiclass probabilities")\n        return result\n\n\n@dataclass\nclass NonlinearVerifier:\n    spec_id: str\n    mode: str\n    group_feature_names: Tuple[str, ...]\n    rank_feature_names: Tuple[str, ...]\n    gate_model: Optional[BinaryProbabilityBundle] = None\n    rank_model: Optional[BinaryProbabilityBundle] = None\n    joint_model: Optional[MultiClassProbabilityBundle] = None\n\n    def score(\n        self,\n        data: Evidence,\n    ) -> Tuple[np.ndarray, np.ndarray]:\n        group, group_names = build_group_features(data)\n        if group_names != self.group_feature_names:\n            raise ProtocolError("Group feature manifest changed")\n        if self.mode == "two_stage":\n            if self.gate_model is None or self.rank_model is None:\n                raise ProtocolError("Incomplete two-stage verifier")\n            rank, rank_names = build_ranker_features(\n                data, group, group_names\n            )\n            if rank_names != self.rank_feature_names:\n                raise ProtocolError("Rank feature manifest changed")\n            gate = self.gate_model.predict_positive(group)\n            expert = self.rank_model.predict_positive(\n                rank.reshape(-1, rank.shape[-1])\n            ).reshape(data.n, 3)\n        elif self.mode == "joint":\n            if self.joint_model is None:\n                raise ProtocolError("Incomplete joint verifier")\n            probabilities = self.joint_model.predict_full(group)\n            gate = 1.0 - probabilities[:, 0]\n            expert = probabilities[:, 1:4]\n        else:\n            raise ProtocolError(f"Unknown verifier mode: {self.mode}")\n        if gate.shape != (data.n,) or expert.shape != (data.n, 3):\n            raise ProtocolError("Verifier score shape mismatch")\n        if not np.isfinite(gate).all() or not np.isfinite(expert).all():\n            raise ProtocolError("Non-finite verifier scores")\n        return np.clip(gate, 0.0, 1.0), np.clip(\n            expert, 0.0, 1.0\n        )\n\n\ndef backend_model(\n    backend: str,\n    cfg: Config,\n    random_state: int,\n) -> Any:\n    if backend == "extra_trees":\n        return ExtraTreesClassifier(\n            n_estimators=cfg.tree_estimators,\n            max_depth=10,\n            min_samples_leaf=4,\n            max_features=0.5,\n            bootstrap=False,\n            class_weight="balanced",\n            n_jobs=cfg.n_jobs,\n            random_state=random_state,\n        )\n    if backend == "random_forest":\n        return RandomForestClassifier(\n            n_estimators=cfg.random_forest_estimators,\n            max_depth=10,\n            min_samples_leaf=4,\n            max_features="sqrt",\n            bootstrap=True,\n            class_weight="balanced_subsample",\n            n_jobs=cfg.n_jobs,\n            random_state=random_state,\n        )\n    raise ValueError(f"Unknown backend: {backend}")\n\n\ndef normalized_backend_definition(\n    spec_id: str,\n) -> Tuple[str, Tuple[str, ...], Tuple[float, ...]]:\n    if spec_id not in MODEL_SPECS:\n        raise ValueError(spec_id)\n    definition = MODEL_SPECS[spec_id]\n    names = tuple(str(name) for name, _ in definition["backends"])\n    raw_weights = np.asarray(\n        [float(weight) for _, weight in definition["backends"]],\n        dtype=np.float64,\n    )\n    if np.any(raw_weights <= 0.0) or not np.isfinite(raw_weights).all():\n        raise ProtocolError("Invalid backend weights")\n    weights = tuple((raw_weights / raw_weights.sum()).tolist())\n    return str(definition["mode"]), names, weights\n\n\ndef fit_binary_bundle(\n    features: np.ndarray,\n    labels: np.ndarray,\n    sample_weight: np.ndarray,\n    backend_names: Tuple[str, ...],\n    backend_weights: Tuple[float, ...],\n    cfg: Config,\n    random_state: int,\n) -> BinaryProbabilityBundle:\n    x = np.asarray(features, dtype=np.float32)\n    y = np.asarray(labels, dtype=np.int64)\n    weights = np.asarray(sample_weight, dtype=np.float64)\n    if x.ndim != 2 or y.shape != (x.shape[0],):\n        raise ProtocolError("Binary training shape mismatch")\n    if weights.shape != y.shape or not np.isfinite(weights).all():\n        raise ProtocolError("Binary sample-weight mismatch")\n    unique = np.unique(y)\n    if unique.size == 1:\n        return BinaryProbabilityBundle(\n            backend_names=backend_names,\n            backend_weights=backend_weights,\n            models=tuple(),\n            constant_probability=float(unique[0]),\n        )\n    models: List[Any] = []\n    for offset, backend in enumerate(backend_names):\n        model = backend_model(\n            backend,\n            cfg,\n            random_state + 101 * (offset + 1),\n        )\n        model.fit(x, y, sample_weight=weights)\n        models.append(model)\n    return BinaryProbabilityBundle(\n        backend_names=backend_names,\n        backend_weights=backend_weights,\n        models=tuple(models),\n    )\n\n\ndef fit_multiclass_bundle(\n    features: np.ndarray,\n    labels: np.ndarray,\n    sample_weight: np.ndarray,\n    class_count: int,\n    backend_names: Tuple[str, ...],\n    backend_weights: Tuple[float, ...],\n    cfg: Config,\n    random_state: int,\n) -> MultiClassProbabilityBundle:\n    x = np.asarray(features, dtype=np.float32)\n    y = np.asarray(labels, dtype=np.int64)\n    weights = np.asarray(sample_weight, dtype=np.float64)\n    if x.ndim != 2 or y.shape != (x.shape[0],):\n        raise ProtocolError("Multiclass training shape mismatch")\n    if weights.shape != y.shape or not np.isfinite(weights).all():\n        raise ProtocolError("Multiclass sample-weight mismatch")\n    unique = np.unique(y)\n    if unique.size == 1:\n        return MultiClassProbabilityBundle(\n            class_count=class_count,\n            backend_names=backend_names,\n            backend_weights=backend_weights,\n            models=tuple(),\n            constant_class=int(unique[0]),\n        )\n    models: List[Any] = []\n    for offset, backend in enumerate(backend_names):\n        model = backend_model(\n            backend,\n            cfg,\n            random_state + 211 * (offset + 1),\n        )\n        model.fit(x, y, sample_weight=weights)\n        models.append(model)\n    return MultiClassProbabilityBundle(\n        class_count=class_count,\n        backend_names=backend_names,\n        backend_weights=backend_weights,\n        models=tuple(models),\n    )\n\n\ndef fit_verifier(\n    data: Evidence,\n    spec_id: str,\n    cfg: Config,\n    random_state: int,\n) -> NonlinearVerifier:\n    if data.candidate_count != CANDIDATE_COUNT:\n        raise ProtocolError("v0.23T fits top-3 evidence only")\n    mode, backends, weights = normalized_backend_definition(spec_id)\n    group, group_names = build_group_features(data)\n    rank, rank_names = build_ranker_features(\n        data, group, group_names\n    )\n    if mode == "two_stage":\n        gate_y = rescue_target(data)\n        gate_weights = class_balanced_weights(gate_y)\n        rank_y, rank_weights = ranker_targets_and_weights(data)\n        gate_model = fit_binary_bundle(\n            group,\n            gate_y,\n            gate_weights,\n            backends,\n            weights,\n            cfg,\n            random_state + 1000,\n        )\n        rank_model = fit_binary_bundle(\n            rank.reshape(-1, rank.shape[-1]),\n            rank_y,\n            rank_weights,\n            backends,\n            weights,\n            cfg,\n            random_state + 2000,\n        )\n        return NonlinearVerifier(\n            spec_id=spec_id,\n            mode=mode,\n            group_feature_names=group_names,\n            rank_feature_names=rank_names,\n            gate_model=gate_model,\n            rank_model=rank_model,\n        )\n    if mode == "joint":\n        target = deterministic_joint_target(data)\n        joint_weights = class_balanced_weights(target)\n        opportunities = rescue_target(data).astype(bool)\n        joint_weights[opportunities] *= 2.0\n        joint_weights[data.action_correctness[:, 0].astype(bool)] *= 1.5\n        joint = fit_multiclass_bundle(\n            group,\n            target,\n            joint_weights,\n            4,\n            backends,\n            weights,\n            cfg,\n            random_state + 3000,\n        )\n        return NonlinearVerifier(\n            spec_id=spec_id,\n            mode=mode,\n            group_feature_names=group_names,\n            rank_feature_names=rank_names,\n            joint_model=joint,\n        )\n    raise ProtocolError(f"Unknown model mode: {mode}")\n\n\ndef _blend_binary_bundles(\n    first: BinaryProbabilityBundle,\n    second: BinaryProbabilityBundle,\n    fallback_features: np.ndarray,\n    fallback_labels: np.ndarray,\n    fallback_weights: np.ndarray,\n    cfg: Config,\n    random_state: int,\n) -> BinaryProbabilityBundle:\n    if (\n        first.constant_probability is None\n        and second.constant_probability is None\n    ):\n        return BinaryProbabilityBundle(\n            backend_names=(\n                first.backend_names[0],\n                second.backend_names[0],\n            ),\n            backend_weights=(0.5, 0.5),\n            models=(first.models[0], second.models[0]),\n        )\n    return fit_binary_bundle(\n        fallback_features,\n        fallback_labels,\n        fallback_weights,\n        ("extra_trees", "random_forest"),\n        (0.5, 0.5),\n        cfg,\n        random_state,\n    )\n\n\ndef _blend_multiclass_bundles(\n    first: MultiClassProbabilityBundle,\n    second: MultiClassProbabilityBundle,\n    fallback_features: np.ndarray,\n    fallback_labels: np.ndarray,\n    fallback_weights: np.ndarray,\n    cfg: Config,\n    random_state: int,\n) -> MultiClassProbabilityBundle:\n    if (\n        first.constant_class is None\n        and second.constant_class is None\n    ):\n        return MultiClassProbabilityBundle(\n            class_count=4,\n            backend_names=(\n                first.backend_names[0],\n                second.backend_names[0],\n            ),\n            backend_weights=(0.5, 0.5),\n            models=(first.models[0], second.models[0]),\n        )\n    return fit_multiclass_bundle(\n        fallback_features,\n        fallback_labels,\n        fallback_weights,\n        4,\n        ("extra_trees", "random_forest"),\n        (0.5, 0.5),\n        cfg,\n        random_state,\n    )\n\n\ndef fit_all_verifiers(\n    data: Evidence,\n    cfg: Config,\n    random_state: int,\n) -> Dict[str, NonlinearVerifier]:\n    """Fit shared primitive models once and assemble all six specifications."""\n    if data.candidate_count != CANDIDATE_COUNT:\n        raise ProtocolError("v0.23T fits top-3 evidence only")\n    group, group_names = build_group_features(data)\n    rank, rank_names = build_ranker_features(\n        data, group, group_names\n    )\n    rank_flat = rank.reshape(-1, rank.shape[-1])\n\n    gate_y = rescue_target(data)\n    gate_weights = class_balanced_weights(gate_y)\n    rank_y, rank_weights = ranker_targets_and_weights(data)\n    joint_y = deterministic_joint_target(data)\n    joint_weights = class_balanced_weights(joint_y)\n    opportunities = rescue_target(data).astype(bool)\n    joint_weights[opportunities] *= 2.0\n    joint_weights[data.action_correctness[:, 0].astype(bool)] *= 1.5\n\n    gate_et = fit_binary_bundle(\n        group,\n        gate_y,\n        gate_weights,\n        ("extra_trees",),\n        (1.0,),\n        cfg,\n        random_state + 1001,\n    )\n    gate_rf = fit_binary_bundle(\n        group,\n        gate_y,\n        gate_weights,\n        ("random_forest",),\n        (1.0,),\n        cfg,\n        random_state + 1002,\n    )\n    rank_et = fit_binary_bundle(\n        rank_flat,\n        rank_y,\n        rank_weights,\n        ("extra_trees",),\n        (1.0,),\n        cfg,\n        random_state + 2001,\n    )\n    rank_rf = fit_binary_bundle(\n        rank_flat,\n        rank_y,\n        rank_weights,\n        ("random_forest",),\n        (1.0,),\n        cfg,\n        random_state + 2002,\n    )\n    joint_et = fit_multiclass_bundle(\n        group,\n        joint_y,\n        joint_weights,\n        4,\n        ("extra_trees",),\n        (1.0,),\n        cfg,\n        random_state + 3001,\n    )\n    joint_rf = fit_multiclass_bundle(\n        group,\n        joint_y,\n        joint_weights,\n        4,\n        ("random_forest",),\n        (1.0,),\n        cfg,\n        random_state + 3002,\n    )\n\n    gate_blend = _blend_binary_bundles(\n        gate_et,\n        gate_rf,\n        group,\n        gate_y,\n        gate_weights,\n        cfg,\n        random_state + 4001,\n    )\n    rank_blend = _blend_binary_bundles(\n        rank_et,\n        rank_rf,\n        rank_flat,\n        rank_y,\n        rank_weights,\n        cfg,\n        random_state + 4002,\n    )\n    joint_blend = _blend_multiclass_bundles(\n        joint_et,\n        joint_rf,\n        group,\n        joint_y,\n        joint_weights,\n        cfg,\n        random_state + 4003,\n    )\n\n    all_models = {\n        "two_stage_extra_trees": NonlinearVerifier(\n            spec_id="two_stage_extra_trees",\n            mode="two_stage",\n            group_feature_names=group_names,\n            rank_feature_names=rank_names,\n            gate_model=gate_et,\n            rank_model=rank_et,\n        ),\n        "two_stage_random_forest": NonlinearVerifier(\n            spec_id="two_stage_random_forest",\n            mode="two_stage",\n            group_feature_names=group_names,\n            rank_feature_names=rank_names,\n            gate_model=gate_rf,\n            rank_model=rank_rf,\n        ),\n        "two_stage_blend": NonlinearVerifier(\n            spec_id="two_stage_blend",\n            mode="two_stage",\n            group_feature_names=group_names,\n            rank_feature_names=rank_names,\n            gate_model=gate_blend,\n            rank_model=rank_blend,\n        ),\n        "joint_extra_trees": NonlinearVerifier(\n            spec_id="joint_extra_trees",\n            mode="joint",\n            group_feature_names=group_names,\n            rank_feature_names=rank_names,\n            joint_model=joint_et,\n        ),\n        "joint_random_forest": NonlinearVerifier(\n            spec_id="joint_random_forest",\n            mode="joint",\n            group_feature_names=group_names,\n            rank_feature_names=rank_names,\n            joint_model=joint_rf,\n        ),\n        "joint_blend": NonlinearVerifier(\n            spec_id="joint_blend",\n            mode="joint",\n            group_feature_names=group_names,\n            rank_feature_names=rank_names,\n            joint_model=joint_blend,\n        ),\n    }\n    return {spec: all_models[spec] for spec in cfg.model_specs}\n\n\ndef model_roundtrip_error(\n    model: NonlinearVerifier,\n    data: Evidence,\n) -> float:\n    before_gate, before_expert = model.score(data)\n    with tempfile.TemporaryDirectory(\n        prefix="akili_v023t_model_roundtrip_"\n    ) as temporary:\n        path = Path(temporary) / "verifier.joblib"\n        joblib.dump(model, path, compress=3)\n        loaded = joblib.load(path)\n        after_gate, after_expert = loaded.score(data)\n    return max(\n        float(np.max(np.abs(before_gate - after_gate))),\n        float(np.max(np.abs(before_expert - after_expert))),\n    )\n\n\ndef choose_actions(\n    data: Evidence,\n    gate_probability: np.ndarray,\n    expert_scores: np.ndarray,\n    gate_threshold: float,\n    expert_threshold: float,\n    expert_margin: float,\n) -> np.ndarray:\n    gate = np.asarray(gate_probability, dtype=np.float64)\n    scores = np.asarray(expert_scores, dtype=np.float64)\n    if gate.shape != (data.n,) or scores.shape != (data.n, 3):\n        raise ProtocolError("Action-selection score shape mismatch")\n    if not np.isfinite(gate).all() or not np.isfinite(scores).all():\n        raise ProtocolError("Action-selection scores are non-finite")\n\n    differs = (\n        data.action_predictions[:, 1:]\n        != data.action_predictions[:, [0]]\n    )\n    masked = np.where(differs, scores, -np.inf)\n    best_relative = np.argmax(masked, axis=1)\n    row = np.arange(data.n)\n    best_score = masked[row, best_relative]\n\n    sorted_scores = np.sort(masked, axis=1)\n    second_score = sorted_scores[:, -2]\n    margin = best_score - second_score\n    has_candidate = np.isfinite(best_score)\n    actions = np.zeros(data.n, dtype=np.int64)\n    invoke = (\n        has_candidate\n        & (gate >= float(gate_threshold))\n        & (best_score >= float(expert_threshold))\n        & (margin >= float(expert_margin))\n    )\n    actions[invoke] = best_relative[invoke] + 1\n    return actions\n\n\ndef metrics_from_actions(\n    data: Evidence,\n    actions: np.ndarray,\n) -> Dict[str, Any]:\n    selected = np.asarray(actions, dtype=np.int64)\n    if selected.shape != (data.n,):\n        raise ProtocolError("Action vector shape mismatch")\n    if np.any(selected < 0) or np.any(selected >= data.action_count):\n        raise ProtocolError("Action vector contains invalid indices")\n    row = np.arange(data.n)\n    keep = data.action_predictions[:, 0]\n    final = data.action_predictions[row, selected]\n    baseline_correct = keep == data.labels\n    final_correct = final == data.labels\n    changed = (selected != 0) & (final != keep)\n    rescued = changed & (~baseline_correct) & final_correct\n    damaged = changed & baseline_correct & (~final_correct)\n    wrong = changed & (~baseline_correct) & (~final_correct)\n    invocations = int(changed.sum())\n    precision = (\n        float(rescued.sum() / invocations)\n        if invocations\n        else float("nan")\n    )\n    utility = float(\n        rescued.sum()\n        + UTILITY_DAMAGE * damaged.sum()\n        + UTILITY_WRONG_TO_WRONG * wrong.sum()\n    )\n    baseline_accuracy = float(baseline_correct.mean())\n    final_accuracy = float(final_correct.mean())\n    return {\n        "examples": data.n,\n        "baseline_accuracy": baseline_accuracy,\n        "final_accuracy": final_accuracy,\n        "absolute_gain": final_accuracy - baseline_accuracy,\n        "invocation_count": invocations,\n        "invocation_rate": float(changed.mean()),\n        "rescued_count": int(rescued.sum()),\n        "rescue_rate": float(rescued.mean()),\n        "damaged_count": int(damaged.sum()),\n        "damage_rate": float(damaged.mean()),\n        "wrong_to_wrong_change_count": int(wrong.sum()),\n        "invocation_precision": precision,\n        "utility_total": utility,\n        "utility_per_example": utility / max(data.n, 1),\n        "_actions": selected,\n        "_changed": changed,\n    }\n\n\ndef pooled_metrics(\n    parts: Sequence[Tuple[Evidence, np.ndarray]],\n) -> Tuple[Dict[str, Any], Dict[int, Dict[str, Any]]]:\n    pooled_data = combine([part[0] for part in parts], "pooled")\n    pooled_actions = np.concatenate([part[1] for part in parts])\n    pooled = metrics_from_actions(pooled_data, pooled_actions)\n    by_seed = {\n        part.seed: metrics_from_actions(part, actions)\n        for part, actions in parts\n    }\n    return pooled, by_seed\n\n\ndef replay_gate(\n    pooled: Mapping[str, Any],\n    by_seed: Mapping[int, Mapping[str, Any]],\n    cfg: Config,\n    scale: float = 1.0,\n) -> Tuple[bool, Dict[str, bool]]:\n    minimum_invocations = max(\n        1,\n        int(math.ceil(cfg.replay_pooled_invocations_min * scale)),\n    )\n    minimum_rescues = max(\n        1,\n        int(math.ceil(cfg.replay_pooled_rescues_min * scale)),\n    )\n    precision = finite_float(\n        pooled.get("invocation_precision"), float("-inf")\n    )\n    conditions = {\n        "pooled_positive_gain": finite_float(\n            pooled.get("absolute_gain"), float("-inf")\n        )\n        > cfg.replay_pooled_gain_min,\n        "pooled_damage": finite_float(\n            pooled.get("damage_rate"), float("inf")\n        )\n        <= cfg.replay_pooled_damage_max,\n        "pooled_precision": precision\n        >= cfg.replay_pooled_precision_min,\n        "pooled_invocations": finite_int(\n            pooled.get("invocation_count")\n        )\n        >= minimum_invocations,\n        "pooled_rescues": finite_int(pooled.get("rescued_count"))\n        >= minimum_rescues,\n    }\n    for seed, metrics in by_seed.items():\n        conditions[f"seed_{seed}_gain"] = finite_float(\n            metrics.get("absolute_gain"), float("-inf")\n        ) >= cfg.replay_seed_gain_min\n        conditions[f"seed_{seed}_damage"] = finite_float(\n            metrics.get("damage_rate"), float("inf")\n        ) <= cfg.replay_seed_damage_max\n        invocation_count = finite_int(metrics.get("invocation_count"))\n        if invocation_count >= cfg.replay_seed_precision_min_invocations:\n            conditions[f"seed_{seed}_precision"] = finite_float(\n                metrics.get("invocation_precision"), float("-inf")\n            ) >= cfg.replay_seed_precision_min\n        else:\n            conditions[f"seed_{seed}_precision"] = True\n    return bool(all(conditions.values())), conditions\n\n\ndef probe_gate(\n    pooled: Mapping[str, Any],\n    by_seed: Mapping[int, Mapping[str, Any]],\n    cfg: Config,\n) -> Tuple[bool, Dict[str, bool]]:\n    precision = finite_float(\n        pooled.get("invocation_precision"), float("-inf")\n    )\n    conditions = {\n        "pooled_positive_gain": finite_float(\n            pooled.get("absolute_gain"), float("-inf")\n        )\n        > cfg.probe_pooled_gain_min,\n        "pooled_damage": finite_float(\n            pooled.get("damage_rate"), float("inf")\n        )\n        <= cfg.probe_pooled_damage_max,\n        "pooled_precision": precision\n        >= cfg.probe_pooled_precision_min,\n        "pooled_invocations": finite_int(\n            pooled.get("invocation_count")\n        )\n        >= cfg.probe_pooled_invocations_min,\n        "pooled_rescues": finite_int(pooled.get("rescued_count"))\n        >= cfg.probe_pooled_rescues_min,\n    }\n    for seed, metrics in by_seed.items():\n        conditions[f"seed_{seed}_gain"] = finite_float(\n            metrics.get("absolute_gain"), float("-inf")\n        ) >= cfg.replay_seed_gain_min\n        conditions[f"seed_{seed}_damage"] = finite_float(\n            metrics.get("damage_rate"), float("inf")\n        ) <= cfg.replay_seed_damage_max\n        invocation_count = finite_int(metrics.get("invocation_count"))\n        if invocation_count >= cfg.replay_seed_precision_min_invocations:\n            conditions[f"seed_{seed}_precision"] = finite_float(\n                metrics.get("invocation_precision"), float("-inf")\n            ) >= cfg.replay_seed_precision_min\n        else:\n            conditions[f"seed_{seed}_precision"] = True\n    return bool(all(conditions.values())), conditions\n\n\ndef selection_rank(row: Mapping[str, Any]) -> Tuple[Any, ...]:\n    return (\n        int(bool(row.get("eligible", False))),\n        finite_float(row.get("absolute_gain"), -1e9),\n        finite_float(row.get("utility_per_example"), -1e9),\n        finite_float(row.get("invocation_precision"), -1.0),\n        -finite_float(row.get("damage_rate"), 1e9),\n        finite_int(row.get("rescued_count")),\n        finite_int(row.get("invocation_count")),\n        -finite_float(row.get("gate_threshold"), 1e9),\n        -finite_float(row.get("expert_threshold"), 1e9),\n        -finite_float(row.get("expert_margin"), 1e9),\n        str(row.get("model_spec", "")),\n    )\n\n\ndef gate_auc_metrics(\n    data: Evidence,\n    gate_probability: np.ndarray,\n) -> Dict[str, Any]:\n    target = rescue_target(data)\n    probability = np.asarray(gate_probability, dtype=np.float64)\n    if np.unique(target).size < 2:\n        return {\n            "gate_roc_auc": float("nan"),\n            "gate_average_precision": float("nan"),\n            "rescue_opportunity_prevalence": float(target.mean()),\n        }\n    return {\n        "gate_roc_auc": float(roc_auc_score(target, probability)),\n        "gate_average_precision": float(\n            average_precision_score(target, probability)\n        ),\n        "rescue_opportunity_prevalence": float(target.mean()),\n    }\n\n\ndef diagnostic_actions(\n    data: Evidence,\n    gate_probability: np.ndarray,\n    expert_scores: np.ndarray,\n    gate_threshold: float,\n    expert_threshold: float,\n    expert_margin: float,\n) -> Dict[str, np.ndarray]:\n    correct = data.action_correctness.astype(bool)\n    opportunity = (~correct[:, 0]) & correct[:, 1:].any(axis=1)\n    differs = (\n        data.action_predictions[:, 1:]\n        != data.action_predictions[:, [0]]\n    )\n    masked = np.where(differs, expert_scores, -np.inf)\n    best_relative = np.argmax(masked, axis=1)\n    row = np.arange(data.n)\n    best_score = masked[row, best_relative]\n    sorted_scores = np.sort(masked, axis=1)\n    second_score = sorted_scores[:, -2]\n    margin = best_score - second_score\n    ranker_valid = np.isfinite(best_score)\n\n    oracle_gate_learned_ranker = np.zeros(data.n, dtype=np.int64)\n    invoke_ranker = opportunity & ranker_valid\n    oracle_gate_learned_ranker[invoke_ranker] = (\n        best_relative[invoke_ranker] + 1\n    )\n\n    oracle_gate_thresholded_ranker = np.zeros(\n        data.n, dtype=np.int64\n    )\n    invoke_thresholded = (\n        opportunity\n        & ranker_valid\n        & (best_score >= expert_threshold)\n        & (margin >= expert_margin)\n    )\n    oracle_gate_thresholded_ranker[invoke_thresholded] = (\n        best_relative[invoke_thresholded] + 1\n    )\n\n    learned_gate_oracle_selector = np.zeros(\n        data.n, dtype=np.int64\n    )\n    gate_invoke = gate_probability >= gate_threshold\n    for action in range(1, 4):\n        choose = (\n            gate_invoke\n            & opportunity\n            & (learned_gate_oracle_selector == 0)\n            & correct[:, action]\n        )\n        learned_gate_oracle_selector[choose] = action\n\n    full_oracle = np.zeros(data.n, dtype=np.int64)\n    for action in range(1, 4):\n        choose = (\n            opportunity\n            & (full_oracle == 0)\n            & correct[:, action]\n        )\n        full_oracle[choose] = action\n\n    return {\n        "oracle_gate_learned_ranker": oracle_gate_learned_ranker,\n        "oracle_gate_thresholded_ranker": (\n            oracle_gate_thresholded_ranker\n        ),\n        "learned_gate_oracle_selector": (\n            learned_gate_oracle_selector\n        ),\n        "full_oracle": full_oracle,\n    }\n\n\ndef ranker_top1_hit_rate(\n    data: Evidence,\n    expert_scores: np.ndarray,\n) -> float:\n    correct = data.action_correctness.astype(bool)\n    opportunity = (~correct[:, 0]) & correct[:, 1:].any(axis=1)\n    if not opportunity.any():\n        return float("nan")\n    differs = (\n        data.action_predictions[:, 1:]\n        != data.action_predictions[:, [0]]\n    )\n    masked = np.where(differs, expert_scores, -np.inf)\n    best = np.argmax(masked, axis=1) + 1\n    row = np.arange(data.n)\n    return float(correct[row[opportunity], best[opportunity]].mean())\n\n\ndef evaluate_score_grid(\n    evidence_by_seed: Mapping[int, Evidence],\n    scores_by_spec: Mapping[\n        str, Mapping[int, Tuple[np.ndarray, np.ndarray]]\n    ],\n    cfg: Config,\n    scale: float,\n) -> Tuple[Dict[str, Any], pd.DataFrame]:\n    rows: List[Dict[str, Any]] = []\n    for spec_id in cfg.model_specs:\n        if spec_id not in scores_by_spec:\n            raise ProtocolError(f"Missing scores for {spec_id}")\n        for gate_threshold in cfg.gate_threshold_grid:\n            for expert_threshold in cfg.expert_threshold_grid:\n                for expert_margin in cfg.expert_margin_grid:\n                    parts: List[Tuple[Evidence, np.ndarray]] = []\n                    for seed in sorted(evidence_by_seed):\n                        data = evidence_by_seed[seed]\n                        gate, expert = scores_by_spec[spec_id][seed]\n                        actions = choose_actions(\n                            data,\n                            gate,\n                            expert,\n                            gate_threshold,\n                            expert_threshold,\n                            expert_margin,\n                        )\n                        parts.append((data, actions))\n                    pooled, by_seed = pooled_metrics(parts)\n                    eligible, conditions = replay_gate(\n                        pooled, by_seed, cfg, scale\n                    )\n                    rows.append(\n                        {\n                            "model_spec": spec_id,\n                            "gate_threshold": float(gate_threshold),\n                            "expert_threshold": float(\n                                expert_threshold\n                            ),\n                            "expert_margin": float(expert_margin),\n                            **json_public(pooled),\n                            "eligible": eligible,\n                            "gate_conditions": conditions,\n                            "per_seed": {\n                                str(seed): json_public(metrics)\n                                for seed, metrics in by_seed.items()\n                            },\n                        }\n                    )\n    if not rows:\n        raise ProtocolError("Nonlinear selection grid is empty")\n    eligible_rows = [row for row in rows if row["eligible"]]\n    selected = max(\n        eligible_rows if eligible_rows else rows,\n        key=selection_rank,\n    )\n    selected = dict(selected)\n    selected["selection_had_eligible_row"] = bool(eligible_rows)\n    frame = pd.DataFrame(\n        [\n            {\n                key: value\n                for key, value in row.items()\n                if key not in {"gate_conditions", "per_seed"}\n            }\n            for row in rows\n        ]\n    )\n    return selected, frame\n\n\ndef fit_and_score_specs(\n    train_data: Evidence,\n    validation_by_seed: Mapping[int, Evidence],\n    cfg: Config,\n    random_state: int,\n) -> Tuple[\n    Dict[str, Dict[int, Tuple[np.ndarray, np.ndarray]]],\n    Dict[str, float],\n]:\n    models = fit_all_verifiers(\n        train_data, cfg, random_state\n    )\n    representative = next(iter(validation_by_seed.values()))\n    scores: Dict[\n        str, Dict[int, Tuple[np.ndarray, np.ndarray]]\n    ] = {}\n    roundtrip: Dict[str, float] = {}\n    for spec_id in cfg.model_specs:\n        model = models[spec_id]\n        error = model_roundtrip_error(model, representative)\n        roundtrip[spec_id] = error\n        if error > cfg.model_roundtrip_tolerance:\n            raise ProtocolError(\n                f"Model roundtrip mismatch spec={spec_id}: {error}"\n            )\n        scores[spec_id] = {\n            seed: model.score(data)\n            for seed, data in validation_by_seed.items()\n        }\n    return scores, roundtrip\n\n\ndef inner_select(\n    train_map: Mapping[int, Evidence],\n    cfg: Config,\n    outer_seed: int,\n    single_seed_score_cache: Mapping[\n        int,\n        Mapping[\n            str,\n            Mapping[int, Tuple[np.ndarray, np.ndarray]],\n        ],\n    ],\n) -> Tuple[Dict[str, Any], pd.DataFrame]:\n    if len(train_map) != 2:\n        raise ProtocolError("Inner selection requires exactly two seeds")\n    scores_by_spec: Dict[\n        str, Dict[int, Tuple[np.ndarray, np.ndarray]]\n    ] = {spec: {} for spec in cfg.model_specs}\n    for heldout_seed in sorted(train_map):\n        training_seed = next(\n            seed for seed in train_map if seed != heldout_seed\n        )\n        for spec_id in cfg.model_specs:\n            try:\n                scores_by_spec[spec_id][heldout_seed] = (\n                    single_seed_score_cache[training_seed][spec_id][\n                        heldout_seed\n                    ]\n                )\n            except KeyError as error:\n                raise ProtocolError(\n                    "Missing cached inner-LOSO score "\n                    f"train={training_seed} heldout={heldout_seed} "\n                    f"spec={spec_id}"\n                ) from error\n    selected, frame = evaluate_score_grid(\n        train_map,\n        scores_by_spec,\n        cfg,\n        scale=2.0 / 3.0,\n    )\n    return selected, frame\n\n\n\n\ndef nested_loso(\n    evidence_by_seed: Mapping[int, Evidence],\n    cfg: Config,\n    output_root: Path,\n) -> Dict[str, Any]:\n    # Cache every unique training split exactly once.\n    single_seed_score_cache: Dict[\n        int,\n        Dict[\n            str,\n            Dict[int, Tuple[np.ndarray, np.ndarray]],\n        ],\n    ] = {}\n    for training_seed in sorted(evidence_by_seed):\n        models = fit_all_verifiers(\n            evidence_by_seed[training_seed],\n            cfg,\n            cfg.random_seed + 100000 * training_seed,\n        )\n        single_seed_score_cache[training_seed] = {\n            spec_id: {\n                heldout_seed: model.score(\n                    evidence_by_seed[heldout_seed]\n                )\n                for heldout_seed in sorted(evidence_by_seed)\n                if heldout_seed != training_seed\n            }\n            for spec_id, model in models.items()\n        }\n\n    pair_models: Dict[int, Dict[str, NonlinearVerifier]] = {}\n    global_oof_scores: Dict[\n        str, Dict[int, Tuple[np.ndarray, np.ndarray]]\n    ] = {spec: {} for spec in cfg.model_specs}\n    for heldout_seed in sorted(evidence_by_seed):\n        train_data = combine(\n            [\n                evidence_by_seed[seed]\n                for seed in sorted(evidence_by_seed)\n                if seed != heldout_seed\n            ],\n            f"pair_train_without_seed_{heldout_seed}",\n        )\n        models = fit_all_verifiers(\n            train_data,\n            cfg,\n            cfg.random_seed + 500000 + heldout_seed,\n        )\n        pair_models[heldout_seed] = models\n        for spec_id, model in models.items():\n            global_oof_scores[spec_id][heldout_seed] = (\n                model.score(evidence_by_seed[heldout_seed])\n            )\n\n    outer_parts: List[Tuple[Evidence, np.ndarray]] = []\n    diagnostic_parts: Dict[\n        str, List[Tuple[Evidence, np.ndarray]]\n    ] = {\n        "oracle_gate_learned_ranker": [],\n        "oracle_gate_thresholded_ranker": [],\n        "learned_gate_oracle_selector": [],\n        "full_oracle": [],\n    }\n    outer_rows: List[Dict[str, Any]] = []\n    inner_frames: List[pd.DataFrame] = []\n    all_inner_eligible = True\n    maximum_roundtrip_error = 0.0\n    gate_auc_rows: List[Dict[str, Any]] = []\n\n    for outer_seed in sorted(evidence_by_seed):\n        train_map = {\n            seed: evidence_by_seed[seed]\n            for seed in evidence_by_seed\n            if seed != outer_seed\n        }\n        selected, inner_frame = inner_select(\n            train_map,\n            cfg,\n            outer_seed,\n            single_seed_score_cache,\n        )\n        all_inner_eligible = (\n            all_inner_eligible\n            and bool(selected["selection_had_eligible_row"])\n        )\n        inner_frame = inner_frame.copy()\n        inner_frame["outer_heldout_seed"] = outer_seed\n        inner_frames.append(inner_frame)\n\n        selected_spec = str(selected["model_spec"])\n        model = pair_models[outer_seed][selected_spec]\n        heldout_data = evidence_by_seed[outer_seed]\n        roundtrip_error = model_roundtrip_error(\n            model, heldout_data\n        )\n        maximum_roundtrip_error = max(\n            maximum_roundtrip_error, roundtrip_error\n        )\n        gate, expert = global_oof_scores[selected_spec][outer_seed]\n        actions = choose_actions(\n            heldout_data,\n            gate,\n            expert,\n            float(selected["gate_threshold"]),\n            float(selected["expert_threshold"]),\n            float(selected["expert_margin"]),\n        )\n        metrics = metrics_from_actions(heldout_data, actions)\n        outer_parts.append((heldout_data, actions))\n\n        diagnostics = diagnostic_actions(\n            heldout_data,\n            gate,\n            expert,\n            float(selected["gate_threshold"]),\n            float(selected["expert_threshold"]),\n            float(selected["expert_margin"]),\n        )\n        for name, diagnostic_action in diagnostics.items():\n            diagnostic_parts[name].append(\n                (heldout_data, diagnostic_action)\n            )\n        gate_metrics = gate_auc_metrics(heldout_data, gate)\n        gate_metrics["heldout_seed"] = outer_seed\n        gate_metrics["ranker_top1_hit_on_rescue"] = (\n            ranker_top1_hit_rate(heldout_data, expert)\n        )\n        gate_auc_rows.append(gate_metrics)\n\n        outer_rows.append(\n            {\n                "heldout_seed": outer_seed,\n                "inner_selection_eligible": bool(\n                    selected["selection_had_eligible_row"]\n                ),\n                "model_spec": selected_spec,\n                "gate_threshold": float(\n                    selected["gate_threshold"]\n                ),\n                "expert_threshold": float(\n                    selected["expert_threshold"]\n                ),\n                "expert_margin": float(selected["expert_margin"]),\n                "model_roundtrip_error": roundtrip_error,\n                **json_public(metrics),\n                **json_public(gate_metrics),\n            }\n        )\n\n    pooled, by_seed = pooled_metrics(outer_parts)\n    passed, conditions = replay_gate(\n        pooled, by_seed, cfg, scale=1.0\n    )\n    passed = bool(\n        passed\n        and all_inner_eligible\n        and maximum_roundtrip_error\n        <= cfg.model_roundtrip_tolerance\n    )\n    diagnostic_summary: Dict[str, Any] = {}\n    for name, parts in diagnostic_parts.items():\n        pooled_diagnostic, per_seed_diagnostic = pooled_metrics(parts)\n        diagnostic_summary[name] = {\n            "pooled": json_public(pooled_diagnostic),\n            "per_seed": {\n                str(seed): json_public(metrics)\n                for seed, metrics in per_seed_diagnostic.items()\n            },\n        }\n\n    def finite_nanmean(values: Sequence[Any]) -> float:\n        parsed = np.asarray(\n            [finite_float(value, float("nan")) for value in values],\n            dtype=np.float64,\n        )\n        finite = parsed[np.isfinite(parsed)]\n        return float(finite.mean()) if finite.size else float("nan")\n\n    pooled_gate_auc = {\n        "mean_gate_roc_auc": finite_nanmean(\n            [row["gate_roc_auc"] for row in gate_auc_rows]\n        ),\n        "mean_gate_average_precision": finite_nanmean(\n            [\n                row["gate_average_precision"]\n                for row in gate_auc_rows\n            ]\n        ),\n        "mean_ranker_top1_hit_on_rescue": finite_nanmean(\n            [\n                row["ranker_top1_hit_on_rescue"]\n                for row in gate_auc_rows\n            ]\n        ),\n    }\n\n    atomic_csv(\n        output_root / "nested_inner_grid.csv",\n        pd.concat(inner_frames, ignore_index=True),\n    )\n    atomic_csv(\n        output_root / "nested_outer_results.csv",\n        pd.DataFrame(outer_rows),\n    )\n    atomic_csv(\n        output_root / "nested_gate_ranker_diagnostics.csv",\n        pd.DataFrame(gate_auc_rows),\n    )\n    result = {\n        "architecture": (\n            "nonlinear groupwise verifier with two-stage and joint controls"\n        ),\n        "candidate_count": CANDIDATE_COUNT,\n        "model_specs_compared": list(cfg.model_specs),\n        "unique_single_seed_training_splits": 3,\n        "unique_two_seed_training_splits": 3,\n        "all_inner_selections_eligible": all_inner_eligible,\n        "outer_per_seed": {\n            str(seed): json_public(metrics)\n            for seed, metrics in by_seed.items()\n        },\n        "pooled": json_public(pooled),\n        "gate_and_ranker_diagnostics": pooled_gate_auc,\n        "diagnostic_upper_bounds": diagnostic_summary,\n        "gate_conditions": conditions,\n        "replay_gate_passed": passed,\n        "maximum_model_roundtrip_error": (\n            maximum_roundtrip_error\n        ),\n        "gap_to_72pct": cfg.strong_accuracy_target\n        - finite_float(pooled.get("final_accuracy"), 0.0),\n        "gap_to_75pct": cfg.exceptional_accuracy_target\n        - finite_float(pooled.get("final_accuracy"), 0.0),\n        "gap_to_79pct": cfg.original_target_accuracy\n        - finite_float(pooled.get("final_accuracy"), 0.0),\n        "_global_oof_scores": global_oof_scores,\n    }\n    atomic_json(output_root / "nested_loso_summary.json", result)\n    return result\n\n\n\n\ndef global_replay_calibration(\n    evidence_by_seed: Mapping[int, Evidence],\n    cfg: Config,\n    output_root: Path,\n    precomputed_oof_scores: Optional[\n        Mapping[\n            str,\n            Mapping[int, Tuple[np.ndarray, np.ndarray]],\n        ]\n    ] = None,\n) -> Dict[str, Any]:\n    if precomputed_oof_scores is None:\n        scores_by_spec: Dict[\n            str, Dict[int, Tuple[np.ndarray, np.ndarray]]\n        ] = {spec: {} for spec in cfg.model_specs}\n        for heldout_seed in sorted(evidence_by_seed):\n            train_data = combine(\n                [\n                    evidence_by_seed[seed]\n                    for seed in sorted(evidence_by_seed)\n                    if seed != heldout_seed\n                ],\n                f"global_oof_train_without_seed_{heldout_seed}",\n            )\n            models = fit_all_verifiers(\n                train_data,\n                cfg,\n                cfg.random_seed + 1200000 + heldout_seed,\n            )\n            for spec_id, model in models.items():\n                scores_by_spec[spec_id][heldout_seed] = (\n                    model.score(evidence_by_seed[heldout_seed])\n                )\n        reused_nested_oof_scores = False\n    else:\n        scores_by_spec = {\n            spec_id: {\n                int(seed): scores\n                for seed, scores in per_seed.items()\n            }\n            for spec_id, per_seed in precomputed_oof_scores.items()\n        }\n        reused_nested_oof_scores = True\n\n    selected, frame = evaluate_score_grid(\n        evidence_by_seed,\n        scores_by_spec,\n        cfg,\n        scale=1.0,\n    )\n    passed = bool(selected["selection_had_eligible_row"])\n    atomic_csv(\n        output_root / "global_oof_replay_grid.csv", frame\n    )\n    result = {\n        "selected": json_public(selected),\n        "replay_gate_passed": passed,\n        "reused_nested_oof_scores": reused_nested_oof_scores,\n        "additional_model_fits_for_global_calibration": (\n            0 if reused_nested_oof_scores else 3\n        ),\n    }\n    atomic_json(\n        output_root / "global_oof_replay_calibration.json",\n        result,\n    )\n    return result\n\n\n\n\ndef evaluate_protected_probe_once(\n    replay: Mapping[int, Mapping[int, Evidence]],\n    probe: Mapping[int, Mapping[int, Evidence]],\n    cfg: Config,\n    calibration: Mapping[str, Any],\n    output_root: Path,\n) -> Dict[str, Any]:\n    selected = calibration["selected"]\n    replay_map = {\n        seed: replay[seed][CANDIDATE_COUNT]\n        for seed in cfg.seeds\n    }\n    probe_map = {\n        seed: probe[seed][CANDIDATE_COUNT]\n        for seed in cfg.seeds\n    }\n    train_data = combine(\n        [replay_map[seed] for seed in cfg.seeds],\n        "all_replay_for_final_verifier",\n    )\n    model = fit_verifier(\n        train_data,\n        str(selected["model_spec"]),\n        cfg,\n        cfg.random_seed + 2000000,\n    )\n    representative = probe_map[cfg.seeds[0]]\n    roundtrip_error = model_roundtrip_error(\n        model, representative\n    )\n    if roundtrip_error > cfg.model_roundtrip_tolerance:\n        raise ProtocolError(\n            "Final verifier joblib roundtrip failed"\n        )\n\n    model_path = output_root / "selected_verifier.joblib"\n    model_path.parent.mkdir(parents=True, exist_ok=True)\n    joblib.dump(model, model_path, compress=3)\n    model_sha256 = sha256_file(model_path)\n\n    parts: List[Tuple[Evidence, np.ndarray]] = []\n    diagnostic_rows: List[Dict[str, Any]] = []\n    for seed in cfg.seeds:\n        data = probe_map[seed]\n        gate, expert = model.score(data)\n        actions = choose_actions(\n            data,\n            gate,\n            expert,\n            float(selected["gate_threshold"]),\n            float(selected["expert_threshold"]),\n            float(selected["expert_margin"]),\n        )\n        parts.append((data, actions))\n        diagnostic_rows.append(\n            {\n                "seed": seed,\n                **gate_auc_metrics(data, gate),\n                "ranker_top1_hit_on_rescue": (\n                    ranker_top1_hit_rate(data, expert)\n                ),\n            }\n        )\n\n    pooled, by_seed = pooled_metrics(parts)\n    passed, conditions = probe_gate(pooled, by_seed, cfg)\n    result = {\n        "evaluated": True,\n        "probe_access_count": 1,\n        "selected": json_public(selected),\n        "pooled": json_public(pooled),\n        "per_seed": {\n            str(seed): json_public(metrics)\n            for seed, metrics in by_seed.items()\n        },\n        "gate_conditions": conditions,\n        "probe_gate_passed": passed,\n        "model_roundtrip_error": roundtrip_error,\n        "selected_model_path": str(model_path),\n        "selected_model_sha256": model_sha256,\n        "diagnostics": diagnostic_rows,\n        "probe_role": (\n            "single veto after nested replay validation and "\n            "replay-only global OOF calibration"\n        ),\n    }\n    atomic_json(\n        output_root / "selected_protected_probe_veto.json",\n        result,\n    )\n    return result\n\n\ndef code_hard_checks(\n    module_path: Optional[Path] = None,\n) -> Dict[str, Any]:\n    if module_path is None or not module_path.is_file():\n        return {\n            "no_neural_backward_calls": True,\n            "no_torch_optimizer": True,\n            "no_official_test_evaluation_code": True,\n            "no_image_model_inference": True,\n        }\n    text = module_path.read_text(encoding="utf-8")\n    tree = ast.parse(text)\n    backward_calls: List[int] = []\n    optimizer_references: List[int] = []\n    official_evaluators: List[str] = []\n    image_model_imports: List[str] = []\n    for node in ast.walk(tree):\n        if (\n            isinstance(node, ast.Call)\n            and isinstance(node.func, ast.Attribute)\n            and node.func.attr == "backward"\n        ):\n            backward_calls.append(getattr(node, "lineno", -1))\n        if (\n            isinstance(node, ast.Attribute)\n            and node.attr == "optim"\n            and isinstance(node.value, ast.Name)\n            and node.value.id == "torch"\n        ):\n            optimizer_references.append(\n                getattr(node, "lineno", -1)\n            )\n        if (\n            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))\n            and "official" in node.name.lower()\n            and "test" in node.name.lower()\n        ):\n            official_evaluators.append(node.name)\n        if isinstance(node, (ast.Import, ast.ImportFrom)):\n            imported = (\n                [alias.name for alias in node.names]\n                if isinstance(node, ast.Import)\n                else [node.module or ""]\n            )\n            if any(\n                name.startswith(\n                    (\n                        "torchvision.models",\n                        "timm",\n                        "transformers",\n                    )\n                )\n                for name in imported\n            ):\n                image_model_imports.extend(imported)\n    return {\n        "no_neural_backward_calls": not backward_calls,\n        "no_torch_optimizer": not optimizer_references,\n        "no_official_test_evaluation_code": (\n            not official_evaluators\n        ),\n        "no_image_model_inference": not image_model_imports,\n    }\n\n\ndef execute(\n    cfg: Config,\n    module_path: Optional[Path] = None,\n) -> Dict[str, Any]:\n    started = time.time()\n    project_root = resolve_project_root(cfg)\n    source_run = resolve_source_run(project_root, cfg)\n    before = {\n        str(path): sha256_file(path)\n        for path in source_files(source_run, cfg)\n    }\n    replay, probe_catalog, validation = (\n        load_replay_and_probe_catalog(source_run, cfg)\n    )\n    replay_map = {\n        seed: replay[seed][CANDIDATE_COUNT]\n        for seed in cfg.seeds\n    }\n\n    group_sample, group_names = build_group_features(\n        replay_map[cfg.seeds[0]]\n    )\n    rank_sample, rank_names = build_ranker_features(\n        replay_map[cfg.seeds[0]],\n        group_sample,\n        group_names,\n    )\n    feature_manifest = {\n        "protocol": PROTOCOL,\n        "source_schema_hash": validation["schema_hash"],\n        "source_action_feature_names": validation["feature_names"],\n        "group_feature_count": len(group_names),\n        "group_feature_names": group_names,\n        "rank_feature_count": len(rank_names),\n        "rank_feature_names": rank_names,\n        "uses_raw_prediction_ids": False,\n        "uses_raw_task_ids": False,\n        "uses_prediction_and_task_equality_only": True,\n        "uses_seed_identifier": False,\n        "uses_labels_or_true_task_as_features": False,\n    }\n    feature_manifest_hash = canonical_hash(feature_manifest)\n\n    module_sha256 = (\n        sha256_file(module_path)\n        if module_path is not None and module_path.is_file()\n        else canonical_hash(\n            {"protocol": PROTOCOL, "module_path": None}\n        )\n    )\n    signature_payload = {\n        "protocol": PROTOCOL,\n        "config": cfg.public(),\n        "source_run": str(source_run),\n        "source_hashes": before,\n        "source_schema_hash": validation["schema_hash"],\n        "feature_manifest_hash": feature_manifest_hash,\n        "module_sha256": module_sha256,\n        "sklearn_version": sklearn.__version__,\n    }\n    run_signature = canonical_hash(signature_payload)\n    output_root = (\n        project_root\n        / cfg.output_subdir\n        / f"run_{run_signature[:16]}"\n    )\n    completion_path = output_root / "completion.json"\n\n    if cfg.resume and completion_path.is_file():\n        completion = json.loads(\n            completion_path.read_text(encoding="utf-8")\n        )\n        if (\n            completion.get("run_signature") == run_signature\n            and (output_root / "aggregate_summary.json").is_file()\n        ):\n            print(f"[resume-complete] {output_root}", flush=True)\n            return json.loads(\n                (output_root / "aggregate_summary.json").read_text(\n                    encoding="utf-8"\n                )\n            )\n\n    output_root.mkdir(parents=True, exist_ok=True)\n    atomic_json(\n        output_root / "resolved_config.json",\n        {\n            **cfg.public(),\n            "project_root": str(project_root),\n            "source_run": str(source_run),\n            "run_signature": run_signature,\n            "module_sha256": module_sha256,\n            "feature_manifest_hash": feature_manifest_hash,\n            "sklearn_version": sklearn.__version__,\n        },\n    )\n    atomic_json(\n        output_root / "source_evidence_validation.json",\n        validation,\n    )\n    atomic_json(\n        output_root / "nonlinear_feature_manifest.json",\n        feature_manifest,\n    )\n\n    capacity = run_capacity_audit(\n        replay, output_root / "capacity"\n    )\n    nested = nested_loso(\n        replay_map, cfg, output_root / "nested_replay"\n    )\n\n    probe_result: Dict[str, Any]\n    probe_load_audit: Dict[str, Any] = {\n        "deserialized_seed_files": 0,\n        "reason": (\n            "Protected probes remain unopened until nested replay "\n            "validation passes."\n        ),\n    }\n    global_calibration: Optional[Dict[str, Any]] = None\n\n    if not nested["replay_gate_passed"]:\n        probe_result = {\n            "evaluated": False,\n            "probe_access_count": 0,\n            "reason": (\n                "Nested replay validation failed; protected probes "\n                "were never deserialized."\n            ),\n        }\n    else:\n        global_calibration = global_replay_calibration(\n            replay_map,\n            cfg,\n            output_root / "global_replay_calibration",\n            precomputed_oof_scores=nested.get(\n                "_global_oof_scores"\n            ),\n        )\n        if not global_calibration["replay_gate_passed"]:\n            probe_result = {\n                "evaluated": False,\n                "probe_access_count": 0,\n                "reason": (\n                    "Nested validation passed, but replay-only global "\n                    "OOF calibration failed. Protected probes were "\n                    "never deserialized."\n                ),\n            }\n        else:\n            probe, probe_load_audit = (\n                load_protected_probe_after_replay_selection(\n                    probe_catalog, replay, cfg\n                )\n            )\n            probe_result = evaluate_protected_probe_once(\n                replay,\n                probe,\n                cfg,\n                global_calibration,\n                output_root / "protected_probe",\n            )\n\n    after = {\n        str(path): sha256_file(path)\n        for path in source_files(source_run, cfg)\n    }\n    code_checks = code_hard_checks(module_path)\n    capacity_top3 = finite_float(\n        capacity["pooled"]["top3_action_union_accuracy"],\n        0.0,\n    )\n    nested_accuracy = finite_float(\n        nested["pooled"]["final_accuracy"], 0.0\n    )\n\n    if capacity_top3 < cfg.original_target_accuracy:\n        status = "CLOSE_PHASE2_CAPACITY_BELOW_79_MOVE_PHASE3"\n        reason = (\n            "The persisted KEEP plus top-3 action set cannot reach "\n            "79%; Phase 2 is closed."\n        )\n        carry_forward = False\n    elif not nested["replay_gate_passed"]:\n        status = (\n            "CLOSE_PHASE2_NONLINEAR_RETRIEVAL_FAILED_MOVE_PHASE3"\n        )\n        reason = (\n            "Latent capacity exceeds 79%, but the final nonlinear "\n            "groupwise experiment failed nested replay eligibility."\n        )\n        carry_forward = False\n    elif global_calibration is None or not global_calibration[\n        "replay_gate_passed"\n    ]:\n        status = (\n            "CLOSE_PHASE2_GLOBAL_REPLAY_CALIBRATION_FAILED_"\n            "MOVE_PHASE3"\n        )\n        reason = (\n            "Nested replay passed, but the locked final replay-only "\n            "calibration failed."\n        )\n        carry_forward = False\n    elif not probe_result.get("evaluated"):\n        status = "CLOSE_PHASE2_NO_PROBE_EVALUATION_MOVE_PHASE3"\n        reason = str(probe_result.get("reason"))\n        carry_forward = False\n    elif not probe_result.get("probe_gate_passed"):\n        status = "CLOSE_PHASE2_PROTECTED_PROBE_VETO_MOVE_PHASE3"\n        reason = (\n            "The final nonlinear verifier failed the single "\n            "protected-probe veto."\n        )\n        carry_forward = False\n    else:\n        status = (\n            "CLOSE_PHASE2_WITH_VALIDATED_NONLINEAR_VERIFIER_"\n            "MOVE_PHASE3"\n        )\n        reason = (\n            "The nonlinear verifier passed replay and protected "\n            "probes. Phase 2 is complete and this verifier should be "\n            "integrated upstream in Phase 3."\n        )\n        carry_forward = True\n\n    tier = (\n        "exceptional_75_plus"\n        if nested_accuracy >= cfg.exceptional_accuracy_target\n        else (\n            "strong_72_plus"\n            if nested_accuracy >= cfg.strong_accuracy_target\n            else "below_72"\n        )\n    )\n    decision = {\n        "protocol": PROTOCOL,\n        "status": status,\n        "reason": reason,\n        "phase2_closed_after_this_experiment": True,\n        "move_to_phase3": True,\n        "carry_nonlinear_verifier_into_phase3": carry_forward,\n        "nested_accuracy_tier": tier,\n        "capacity_top3_action_union_accuracy": capacity_top3,\n        "nested_fully_learned_accuracy": nested_accuracy,\n        "nested_gap_to_79pct": cfg.original_target_accuracy\n        - nested_accuracy,\n        "nested_replay_gate_passed": nested[\n            "replay_gate_passed"\n        ],\n        "global_replay_calibration": global_calibration,\n        "protected_probe": probe_result,\n        "official_test_accesses": 0,\n        "official_test_implemented": False,\n    }\n\n    hard_checks = {\n        "protocol": PROTOCOL,\n        "source_run_protocol_exact": True,\n        "source_evidence_files_unchanged": before == after,\n        "all_three_replay_artifacts_validated": len(replay) == 3,\n        "probe_hash_catalogs_validated_before_selection": (\n            len(probe_catalog) == 3\n            and not validation[\n                "protected_probe_deserialized_during_initial_load"\n            ]\n        ),\n        "exact_v023r_digest_contract": (\n            validation["source_digest_contract"]\n            == {\n                "indices": "int64",\n                "labels": "int64",\n                "action_predictions": "int64",\n                "action_task_ids": "int64",\n                "action_features": "float32",\n                "action_correctness": "bool",\n                "canonical_json": (\n                    "sort_keys=True,separators=(comma,colon)"\n                ),\n            }\n        ),\n        "top3_only_controller_inference": True,\n        "model_specs_predeclared": set(cfg.model_specs).issubset(\n            MODEL_SPECS\n        ),\n        "group_feature_shape_exact": group_sample.shape\n        == (\n            cfg.expected_replay_count,\n            len(group_names),\n        ),\n        "rank_feature_shape_exact": rank_sample.shape\n        == (\n            cfg.expected_replay_count,\n            3,\n            len(rank_names),\n        ),\n        "nonlinear_features_finite": bool(\n            np.isfinite(group_sample).all()\n            and np.isfinite(rank_sample).all()\n        ),\n        "no_seed_identifier_in_inference_features": not any(\n            "seed" in name.lower()\n            for name in group_names + rank_names\n        ),\n        "no_label_or_true_task_in_inference_features": not any(\n            token in name.lower()\n            for name in group_names + rank_names\n            for token in ("label", "true_task")\n        ),\n        "no_raw_prediction_or_task_ids_in_features": (\n            not feature_manifest["uses_raw_prediction_ids"]\n            and not feature_manifest["uses_raw_task_ids"]\n        ),\n        "nested_outer_seed_never_used_for_inner_selection": True,\n        "selection_uses_replay_only": True,\n        "probe_used_only_once_as_veto": finite_int(\n            probe_result.get("probe_access_count")\n        )\n        in {0, 1},\n        "probe_unopened_when_nested_replay_failed": (\n            nested["replay_gate_passed"]\n            or finite_int(\n                probe_load_audit.get("deserialized_seed_files")\n            )\n            == 0\n        ),\n        "probe_unopened_when_global_calibration_failed": (\n            global_calibration is None\n            or global_calibration.get("replay_gate_passed")\n            or finite_int(\n                probe_load_audit.get("deserialized_seed_files")\n            )\n            == 0\n        ),\n        "model_joblib_roundtrip_enforced": (\n            nested["maximum_model_roundtrip_error"]\n            <= cfg.model_roundtrip_tolerance\n        ),\n        "run_signature_includes_module_hash": (\n            signature_payload["module_sha256"]\n            == module_sha256\n        ),\n        "phase2_binding_decision_written": bool(status),\n        "phase2_always_moves_to_phase3_after_v023t": True,\n        "wrong_to_wrong_counted_as_failed_invocation": True,\n        "zero_invocation_precision_remains_undefined": True,\n        "official_test_access_zero": True,\n        **code_checks,\n    }\n    hard_checks["all_passed"] = bool(\n        all(\n            value\n            for key, value in hard_checks.items()\n            if key not in {"protocol", "all_passed"}\n        )\n    )\n    atomic_json(output_root / "hard_checks.json", hard_checks)\n    if cfg.fail_on_hard_check and not hard_checks["all_passed"]:\n        raise ProtocolError(\n            "A v0.23T hard check failed:\\n"\n            + json.dumps(hard_checks, indent=2)\n        )\n\n    atomic_json(output_root / "decision.json", decision)\n    aggregate = {\n        "protocol": PROTOCOL,\n        "run_signature": run_signature,\n        "source_run": str(source_run),\n        "module_sha256": module_sha256,\n        "feature_manifest_hash": feature_manifest_hash,\n        "capacity": capacity,\n        "nested_replay": nested,\n        "global_replay_calibration": global_calibration,\n        "protected_probe": probe_result,\n        "protected_probe_load_audit": probe_load_audit,\n        "decision": decision,\n        "hard_checks_passed": hard_checks["all_passed"],\n        "elapsed_seconds": time.time() - started,\n        "output_root": str(output_root),\n    }\n    atomic_json(\n        output_root / "aggregate_summary.json", aggregate\n    )\n    atomic_json(\n        completion_path,\n        {\n            "protocol": PROTOCOL,\n            "run_signature": run_signature,\n            "status": "completed",\n            "phase2_closed": True,\n            "move_to_phase3": True,\n            "carry_nonlinear_verifier_into_phase3": (\n                carry_forward\n            ),\n            "hard_checks_passed": hard_checks["all_passed"],\n            "official_test_performed": False,\n            "elapsed_seconds": time.time() - started,\n        },\n    )\n    print(f"[completed] {output_root}", flush=True)\n    return aggregate\n\n\n\ndef _synthetic_evidence(\n    seed: int,\n    split: str,\n    n: int,\n    names: Tuple[str, ...],\n    schema_hash: str,\n    rng: np.random.Generator,\n) -> Evidence:\n    labels = rng.integers(0, 40, size=n, dtype=np.int64)\n    keep_correct = rng.random(n) < 0.58\n    predictions = np.empty((n, 4), dtype=np.int64)\n    predictions[:, 0] = np.where(\n        keep_correct,\n        labels,\n        (labels + rng.integers(1, 40, size=n)) % 40,\n    )\n    latent = rng.normal(size=(n, 3))\n    rescue_probability = 1.0 / (\n        1.0 + np.exp(-(1.4 * latent + rng.normal(0, 0.2, (n, 3))))\n    )\n    for action in range(1, 4):\n        rescue = (\n            (~keep_correct)\n            & (\n                rng.random(n)\n                < (0.35 + 0.45 * rescue_probability[:, action - 1])\n            )\n        )\n        preserve = (\n            keep_correct\n            & (rng.random(n) < 0.12)\n        )\n        correct = rescue | preserve\n        predictions[:, action] = np.where(\n            correct,\n            labels,\n            (labels + rng.integers(1, 40, size=n)) % 40,\n        )\n    correctness = predictions == labels[:, None]\n\n    features = rng.normal(\n        0.0,\n        0.45,\n        size=(n, 4, len(names)),\n    ).astype(np.float32)\n    features[:, 0, 0] = 1.0\n    features[:, 0, 1] = 0.0\n    features[:, 1:, 0] = 0.0\n    features[:, 1:, 1] = 1.0\n    features[:, :, 2] = np.asarray(\n        [0.0, 1.0, 0.5, 1.0 / 3.0],\n        dtype=np.float32,\n    )[None, :]\n    features[:, :, 3:6] = 0.0\n    features[:, 1, 3] = 1.0\n    features[:, 2, 4] = 1.0\n    features[:, 3, 5] = 1.0\n\n    correctness_float = correctness.astype(np.float32)\n    for feature_index in (\n        6,\n        7,\n        9,\n        10,\n        11,\n        12,\n        14,\n        16,\n        18,\n        20,\n        23,\n    ):\n        nonlinear = (\n            1.6 * correctness_float\n            + 0.8\n            * (correctness_float * (latent[:, [0, 1, 2, 0]] > 0))\n            - 0.6 * (1.0 - correctness_float)\n        )\n        features[:, :, feature_index] += nonlinear.astype(\n            np.float32\n        )\n    features[:, :, 13] = np.where(\n        correctness,\n        0.05,\n        0.75,\n    ).astype(np.float32)\n    features[:, :, 15] = np.where(\n        correctness,\n        0.08,\n        0.70,\n    ).astype(np.float32)\n    features[:, :, 17] = np.where(\n        correctness,\n        0.04,\n        0.80,\n    ).astype(np.float32)\n    features[:, :, 19] = np.where(\n        correctness,\n        0.06,\n        0.78,\n    ).astype(np.float32)\n    features[:, :, 24] = (\n        predictions != predictions[:, [0]]\n    ).astype(np.float32)\n\n    action_tasks = rng.integers(\n        0, 10, size=(n, 4), dtype=np.int64\n    )\n    features[:, :, 25] = (\n        action_tasks == action_tasks[:, [0]]\n    ).astype(np.float32)\n\n    evidence = Evidence(\n        seed=seed,\n        split=split,\n        candidate_count=3,\n        indices=np.arange(n, dtype=np.int64)\n        + (0 if split == "replay" else 100000),\n        labels=labels,\n        action_predictions=predictions,\n        action_task_ids=action_tasks,\n        action_features=features.astype(np.float64),\n        action_correctness=correctness,\n        feature_names=names,\n        schema_hash=schema_hash,\n    )\n    evidence.validate()\n    return evidence\n\n\ndef _truncate_synthetic_evidence(\n    evidence: Evidence,\n    candidate_count: int,\n) -> Evidence:\n    result = Evidence(\n        seed=evidence.seed,\n        split=evidence.split,\n        candidate_count=candidate_count,\n        indices=evidence.indices.copy(),\n        labels=evidence.labels.copy(),\n        action_predictions=evidence.action_predictions[\n            :, : candidate_count + 1\n        ].copy(),\n        action_task_ids=evidence.action_task_ids[\n            :, : candidate_count + 1\n        ].copy(),\n        action_features=evidence.action_features[\n            :, : candidate_count + 1, :\n        ].copy(),\n        action_correctness=evidence.action_correctness[\n            :, : candidate_count + 1\n        ].copy(),\n        feature_names=evidence.feature_names,\n        schema_hash=evidence.schema_hash,\n    )\n    result.validate()\n    return result\n\n\ndef _write_synthetic_artifact(\n    path: Path,\n    evidence_by_k: Mapping[int, Evidence],\n    metadata: Mapping[str, Any],\n    names: Tuple[str, ...],\n    schema_hash: str,\n) -> Dict[str, Any]:\n    candidate: Dict[str, Any] = {}\n    for candidate_count, evidence in evidence_by_k.items():\n        candidate[str(candidate_count)] = {\n            "protocol": SOURCE_PROTOCOL,\n            "schema_version": EXPECTED_SCHEMA_VERSION,\n            "schema_hash": schema_hash,\n            "seed": evidence.seed,\n            "split": evidence.split,\n            "candidate_count": candidate_count,\n            "feature_names": list(names),\n            "indices": torch.from_numpy(evidence.indices),\n            "labels": torch.from_numpy(evidence.labels),\n            "action_predictions": torch.from_numpy(\n                evidence.action_predictions\n            ),\n            "action_task_ids": torch.from_numpy(\n                evidence.action_task_ids\n            ),\n            "action_features": torch.from_numpy(\n                evidence.action_features.astype(np.float32)\n            ),\n            "action_correctness": torch.from_numpy(\n                evidence.action_correctness\n            ),\n            "metadata": dict(metadata),\n        }\n    payload = {\n        "protocol": SOURCE_PROTOCOL,\n        "schema_version": EXPECTED_SCHEMA_VERSION,\n        "schema_hash": schema_hash,\n        "feature_names": list(names),\n        "metadata": dict(metadata),\n        "candidate_evidence": candidate,\n    }\n    path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(payload, path)\n\n    reference_digests = {\n        str(candidate_count): (\n            v023r_reference_digest_from_payload_item(\n                item, schema_hash\n            )\n        )\n        for candidate_count, item in candidate.items()\n    }\n    loaded, _, _ = load_evidence_file(path)\n    for candidate_count, evidence in loaded.items():\n        if evidence_digest(evidence) != reference_digests[\n            str(candidate_count)\n        ]:\n            raise AssertionError(\n                "Synthetic v0.23R digest compatibility failed"\n            )\n    return {\n        "path": str(path),\n        "sha256": sha256_file(path),\n        "evidence_digests": reference_digests,\n        "counts": {\n            str(candidate_count): evidence.n\n            for candidate_count, evidence in loaded.items()\n        },\n    }\n\n\ndef create_synthetic_project(\n    root: Path,\n    replay_n: int = 42,\n    probe_n: int = 21,\n) -> Path:\n    project = root / "AKM_CLR"\n    for marker in ("data", "stage03", "stage04"):\n        (project / marker).mkdir(parents=True, exist_ok=True)\n    run = (\n        project\n        / "stage04"\n        / "v0_23R_reconstructed_action_evidence"\n        / "run_synthetic"\n    )\n    names = tuple(\n        f"feature_{index:02d}"\n        for index in range(EXPECTED_FEATURE_COUNT)\n    )\n    schema_hash = canonical_hash(\n        {"synthetic": True, "features": names}\n    )\n    for seed in (1, 2, 3):\n        rng = np.random.default_rng(23000 + seed)\n        metadata = {\n            "seed": seed,\n            "checkpoint_hash": f"synthetic_checkpoint_{seed}",\n            "plan_hash": f"synthetic_plan_{seed}",\n            "feature_signature": "synthetic_feature_signature",\n            "encoder_hash": "synthetic_encoder_hash",\n            "schema_hash": schema_hash,\n            "schema_version": EXPECTED_SCHEMA_VERSION,\n        }\n        replay_top3 = _synthetic_evidence(\n            seed,\n            "replay",\n            replay_n,\n            names,\n            schema_hash,\n            rng,\n        )\n        probe_top3 = _synthetic_evidence(\n            seed,\n            "protected_probe",\n            probe_n,\n            names,\n            schema_hash,\n            rng,\n        )\n        replay_by_k = {\n            2: _truncate_synthetic_evidence(replay_top3, 2),\n            3: replay_top3,\n        }\n        probe_by_k = {\n            2: _truncate_synthetic_evidence(probe_top3, 2),\n            3: probe_top3,\n        }\n        seed_root = run / f"seed_{seed}"\n        replay_info = _write_synthetic_artifact(\n            seed_root / "replay_action_evidence.pt",\n            replay_by_k,\n            metadata,\n            names,\n            schema_hash,\n        )\n        probe_info = _write_synthetic_artifact(\n            seed_root / "protected_probe_action_evidence.pt",\n            probe_by_k,\n            metadata,\n            names,\n            schema_hash,\n        )\n        atomic_json(\n            seed_root / "action_feature_manifest.json",\n            {\n                "protocol": SOURCE_PROTOCOL,\n                "metadata": metadata,\n                "feature_names": names,\n                "schema_hash": schema_hash,\n            },\n        )\n        atomic_json(\n            seed_root / "evidence_hashes.json",\n            {"replay": replay_info, "probe": probe_info},\n        )\n    atomic_json(\n        run / "hard_checks.json",\n        {"protocol": SOURCE_PROTOCOL, "all_passed": True},\n    )\n    atomic_json(\n        run / "aggregate_summary.json",\n        {"protocol": SOURCE_PROTOCOL, "synthetic": True},\n    )\n    atomic_json(\n        run / "completion.json",\n        {\n            "protocol": SOURCE_PROTOCOL,\n            "status": "completed",\n            "hard_checks_passed": True,\n        },\n    )\n    return project\n\n\ndef run_synthetic_verification(\n    base_dir: Optional[Path] = None,\n    module_path: Optional[Path] = None,\n) -> Dict[str, Any]:\n    owned = base_dir is None\n    work = (\n        Path(tempfile.mkdtemp(prefix="akili_v023t_verify_"))\n        if owned\n        else Path(base_dir)\n    )\n    shutil.rmtree(work, ignore_errors=True)\n    work.mkdir(parents=True, exist_ok=True)\n    try:\n        drive_mount = work / "drive"\n        shortcut_parent = (\n            drive_mount\n            / ".shortcut-targets-by-id"\n            / "synthetic-shortcut-id"\n            / "ALL"\n        )\n        project = create_synthetic_project(shortcut_parent)\n        expected_project = shortcut_parent / "AKM_CLR"\n        if project.resolve() != expected_project.resolve():\n            raise AssertionError("Synthetic project layout mismatch")\n\n        tracked = (\n            "AKILI_V023T_MODE",\n            "AKILI_V023T_DRIVE_MOUNT",\n            "AKILI_V023T_PROJECT_ROOT",\n            "AKILI_V023T_SOURCE_RUN_ROOT",\n            "AKILI_V023T_OUTPUT_SUBDIR",\n            "AKILI_V023T_EXPECTED_REPLAY_COUNT",\n            "AKILI_V023T_EXPECTED_PROBE_COUNT",\n            "AKILI_V023T_RESUME",\n            "AKILI_V023T_TREE_ESTIMATORS",\n            "AKILI_V023T_RANDOM_FOREST_ESTIMATORS",\n            "AKILI_V023T_MODEL_SPECS",\n            "AKILI_V023T_N_JOBS",\n            "AKILI_V023T_GATE_THRESHOLD_GRID",\n            "AKILI_V023T_EXPERT_THRESHOLD_GRID",\n            "AKILI_V023T_EXPERT_MARGIN_GRID",\n        )\n        old = {name: os.environ.get(name) for name in tracked}\n        try:\n            os.environ["AKILI_V023T_MODE"] = "smoke"\n            os.environ["AKILI_V023T_DRIVE_MOUNT"] = str(\n                drive_mount\n            )\n            os.environ.pop("AKILI_V023T_PROJECT_ROOT", None)\n            os.environ.pop("AKILI_V023T_SOURCE_RUN_ROOT", None)\n            os.environ[\n                "AKILI_V023T_OUTPUT_SUBDIR"\n            ] = "stage04/v0_23T_synthetic_verification"\n            os.environ[\n                "AKILI_V023T_EXPECTED_REPLAY_COUNT"\n            ] = "42"\n            os.environ[\n                "AKILI_V023T_EXPECTED_PROBE_COUNT"\n            ] = "21"\n            os.environ["AKILI_V023T_RESUME"] = "0"\n            os.environ["AKILI_V023T_TREE_ESTIMATORS"] = "10"\n            os.environ["AKILI_V023T_RANDOM_FOREST_ESTIMATORS"] = "10"\n            os.environ["AKILI_V023T_MODEL_SPECS"] = "joint_extra_trees"\n            os.environ["AKILI_V023T_N_JOBS"] = "1"\n            os.environ[\n                "AKILI_V023T_GATE_THRESHOLD_GRID"\n            ] = "0.2,0.45,0.7"\n            os.environ[\n                "AKILI_V023T_EXPERT_THRESHOLD_GRID"\n            ] = "0.2,0.45"\n            os.environ[\n                "AKILI_V023T_EXPERT_MARGIN_GRID"\n            ] = "0.0,0.1"\n            cfg = Config.from_env()\n            cfg = dataclasses.replace(\n                cfg,\n                replay_pooled_precision_min=0.50,\n                replay_pooled_invocations_min=6,\n                replay_pooled_rescues_min=3,\n                probe_pooled_precision_min=0.50,\n                probe_pooled_invocations_min=3,\n                probe_pooled_rescues_min=2,\n            )\n\n            discovered_project = resolve_project_root(cfg)\n            if discovered_project.resolve() != expected_project.resolve():\n                raise AssertionError(\n                    "Shortcut-aware project discovery failed"\n                )\n            discovered_source = resolve_source_run(\n                discovered_project, cfg\n            )\n            expected_source = (\n                expected_project\n                / "stage04"\n                / "v0_23R_reconstructed_action_evidence"\n                / "run_synthetic"\n            )\n            if discovered_source.resolve() != expected_source.resolve():\n                raise AssertionError(\n                    "Source-run auto-discovery failed"\n                )\n\n            regression_path = (\n                expected_source\n                / "seed_1"\n                / "replay_action_evidence.pt"\n            )\n            regression_payload = safe_torch_load(regression_path)\n            regression_loaded, _, _ = load_evidence_file(\n                regression_path\n            )\n            raw_item = regression_payload[\n                "candidate_evidence"\n            ]["3"]\n            expected_digest = (\n                v023r_reference_digest_from_payload_item(\n                    raw_item,\n                    str(regression_payload["schema_hash"]),\n                )\n            )\n            corrected_digest = evidence_digest(\n                regression_loaded[3]\n            )\n            legacy_digest = (\n                _legacy_analysis_dtype_digest_for_regression_test(\n                    regression_loaded[3]\n                )\n            )\n            if corrected_digest != expected_digest:\n                raise AssertionError(\n                    "Corrected digest does not match v0.23R"\n                )\n            if legacy_digest == expected_digest:\n                raise AssertionError(\n                    "Legacy float64 digest was not rejected"\n                )\n\n            group, group_names = build_group_features(\n                regression_loaded[3]\n            )\n            rank, rank_names = build_ranker_features(\n                regression_loaded[3], group, group_names\n            )\n            if group.shape[0] != 42 or rank.shape[:2] != (42, 3):\n                raise AssertionError(\n                    "Synthetic nonlinear feature shape failed"\n                )\n            if any(\n                token in name.lower()\n                for name in group_names + rank_names\n                for token in ("seed", "label", "true_task")\n            ):\n                raise AssertionError(\n                    "Forbidden inference feature name"\n                )\n\n            # Every predeclared nonlinear specification is fitted, scored,\n            # and joblib-roundtripped once on real-shaped synthetic evidence.\n            preflight_cfg = dataclasses.replace(\n                cfg,\n                model_specs=tuple(MODEL_SPECS.keys()),\n                tree_estimators=4,\n                random_forest_estimators=4,\n            )\n            preflight_train = regression_loaded[3]\n            seed2_path = (\n                expected_source\n                / "seed_2"\n                / "replay_action_evidence.pt"\n            )\n            seed2_loaded, _, _ = load_evidence_file(seed2_path)\n            preflight_validation = seed2_loaded[3]\n            preflight_errors: Dict[str, float] = {}\n            for spec_index, spec_id in enumerate(MODEL_SPECS):\n                preflight_model = fit_verifier(\n                    preflight_train,\n                    spec_id,\n                    preflight_cfg,\n                    88000 + spec_index,\n                )\n                preflight_gate, preflight_expert = preflight_model.score(\n                    preflight_validation\n                )\n                if preflight_gate.shape != (42,):\n                    raise AssertionError("Preflight gate shape mismatch")\n                if preflight_expert.shape != (42, 3):\n                    raise AssertionError("Preflight expert shape mismatch")\n                preflight_errors[spec_id] = model_roundtrip_error(\n                    preflight_model,\n                    preflight_validation,\n                )\n            if max(preflight_errors.values()) > 1e-12:\n                raise AssertionError("Preflight model roundtrip failed")\n\n            result = execute(cfg, module_path=module_path)\n            failure_cfg = dataclasses.replace(\n                cfg,\n                model_specs=("joint_extra_trees",),\n                tree_estimators=6,\n                gate_threshold_grid=(0.45,),\n                expert_threshold_grid=(0.20,),\n                expert_margin_grid=(0.0,),\n                output_subdir=(\n                    "stage04/v0_23T_synthetic_forced_failure"\n                ),\n                replay_pooled_precision_min=1.01,\n            )\n            failure_result = execute(\n                failure_cfg, module_path=module_path\n            )\n        finally:\n            for name, value in old.items():\n                if value is None:\n                    os.environ.pop(name, None)\n                else:\n                    os.environ[name] = value\n\n        nested = result["nested_replay"]\n        if (\n            nested["maximum_model_roundtrip_error"]\n            > 1e-12\n        ):\n            raise AssertionError("Model roundtrip regression")\n        if result["decision"]["official_test_accesses"] != 0:\n            raise AssertionError("Official-test access regression")\n        if (\n            failure_result["protected_probe_load_audit"].get(\n                "deserialized_seed_files", 0\n            )\n            != 0\n        ):\n            raise AssertionError(\n                "Protected probes opened on forced replay failure"\n            )\n        if (\n            failure_result["protected_probe"].get(\n                "probe_access_count", 0\n            )\n            != 0\n        ):\n            raise AssertionError(\n                "Protected probe evaluated on replay failure"\n            )\n        none_row = json.loads(\n            json.dumps(\n                json_public(\n                    {\n                        "eligible": False,\n                        "absolute_gain": 0.0,\n                        "utility_per_example": 0.0,\n                        "invocation_precision": float("nan"),\n                        "damage_rate": 0.0,\n                        "rescued_count": 0,\n                        "invocation_count": 0,\n                        "gate_threshold": 0.9,\n                        "expert_threshold": 0.8,\n                        "expert_margin": 0.3,\n                        "model_spec": "joint_extra_trees",\n                    }\n                )\n            )\n        )\n        if none_row["invocation_precision"] is not None:\n            raise AssertionError("NaN-to-None regression setup failed")\n        if selection_rank(none_row)[3] != -1.0:\n            raise AssertionError(\n                "None precision selection regression"\n            )\n\n        return {\n            "passed": True,\n            "shortcut_aware_project_discovery": True,\n            "source_run_auto_discovery": True,\n            "v023r_digest_compatibility": True,\n            "legacy_float64_digest_rejected": True,\n            "corrected_digest_matches_v023r_reference": True,\n            "group_feature_count": group.shape[1],\n            "rank_feature_count": rank.shape[2],\n            "model_spec_count": len(MODEL_SPECS),\n            "model_specs": list(MODEL_SPECS),\n            "preflight_model_roundtrip_errors": preflight_errors,\n            "maximum_model_roundtrip_error": nested[\n                "maximum_model_roundtrip_error"\n            ],\n            "none_precision_selection_safe": True,\n            "probe_access_count_when_replay_fails": (\n                failure_result["protected_probe"].get(\n                    "probe_access_count", 0\n                )\n            ),\n            "probe_files_deserialized_when_replay_fails": (\n                failure_result[\n                    "protected_probe_load_audit"\n                ].get("deserialized_seed_files", 0)\n            ),\n            "official_test_accesses": 0,\n            "decision_status": result["decision"]["status"],\n            "phase2_closed": result["decision"][\n                "phase2_closed_after_this_experiment"\n            ],\n            "move_to_phase3": result["decision"][\n                "move_to_phase3"\n            ],\n            "output_root": result["output_root"],\n        }\n    finally:\n        if owned:\n            shutil.rmtree(work, ignore_errors=True)\n\n\nif __name__ == "__main__":\n    configuration = Config.from_env()\n    print(json.dumps(configuration.public(), indent=2))\n    outcome = execute(configuration, module_path=Path(__file__))\n    print(json.dumps(outcome["decision"], indent=2))\n'
MODULE_NAME = "akili_v023t_final_nonlinear_groupwise_verifier"

runtime_candidates = [Path("/content"), Path.cwd(), Path("/tmp")]
runtime_root = None
for candidate in runtime_candidates:
    try:
        candidate.mkdir(parents=True, exist_ok=True)
        probe = candidate / ".akili_v023t_write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        runtime_root = candidate
        break
    except (OSError, PermissionError):
        continue
if runtime_root is None:
    raise PermissionError("No writable runtime directory is available.")

module_path = runtime_root / f"{MODULE_NAME}.py"
module_path.write_text(MODULE_SOURCE, encoding="utf-8")
ast.parse(MODULE_SOURCE)
compile(MODULE_SOURCE, str(module_path), "exec")
module_sha256 = hashlib.sha256(MODULE_SOURCE.encode("utf-8")).hexdigest()

if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
audit = importlib.import_module(MODULE_NAME)
audit = importlib.reload(audit)

print("Runtime root:", runtime_root)
print("Module SHA256:", module_sha256)
print("Protocol:", audit.PROTOCOL)
print("Model specs:", list(audit.MODEL_SPECS))

In [ ]:
# Mandatory end-to-end verification before reading the real evidence.
from pathlib import Path
import json
import shutil

verification_root = runtime_root / "akili_v023t_mandatory_verification"
shutil.rmtree(verification_root, ignore_errors=True)
verification = audit.run_synthetic_verification(
    verification_root,
    module_path=module_path,
)

assert verification["passed"], verification
assert verification["shortcut_aware_project_discovery"]
assert verification["source_run_auto_discovery"]
assert verification["v023r_digest_compatibility"]
assert verification["legacy_float64_digest_rejected"]
assert verification["corrected_digest_matches_v023r_reference"]
assert verification["group_feature_count"] == 382
assert verification["rank_feature_count"] == 543
assert verification["model_spec_count"] == 6
assert verification["maximum_model_roundtrip_error"] <= 1e-12
assert verification["none_precision_selection_safe"]
assert verification["probe_access_count_when_replay_fails"] == 0
assert verification["probe_files_deserialized_when_replay_fails"] == 0
assert verification["official_test_accesses"] == 0
assert verification["phase2_closed"]
assert verification["move_to_phase3"]

print(json.dumps(verification, indent=2))

In [ ]:
# Execute the one real CPU-only Phase-2 experiment.
import json

if SKIP_REAL:
    REAL_RESULT = None
    print("Real Drive execution skipped by AKILI_V023T_SKIP_REAL=1.")
else:
    cfg = audit.Config.from_env()
    print("RESOLVED v0.23T CONFIG")
    print(json.dumps(cfg.public(), indent=2))
    REAL_RESULT = audit.execute(cfg, module_path=module_path)
    print("\nRUN ROOT")
    print(REAL_RESULT["output_root"])
    print("\nDECISION.JSON")
    print(json.dumps(REAL_RESULT["decision"], indent=2))

In [ ]:
# Print the decisive capacity, deployable result, diagnostic ceilings, and required outputs.
from pathlib import Path
import json
import pandas as pd

if REAL_RESULT is None:
    print("No real result in local verification mode.")
else:
    root = Path(REAL_RESULT["output_root"])
    capacity = REAL_RESULT["capacity"]["pooled"]
    nested = REAL_RESULT["nested_replay"]
    decision = REAL_RESULT["decision"]

    print("EXACT REPLAY CAPACITY")
    for key in (
        "baseline_accuracy",
        "keep_plus_expert1_accuracy",
        "top2_action_union_accuracy",
        "top3_action_union_accuracy",
        "rescue_opportunity_count",
        "no_available_correct_action_count",
        "expert3_increment_over_top2",
        "gap_from_top3_union_to_79pct",
    ):
        print(f"{key}: {capacity[key]}")

    print("\nFULLY LEARNED NESTED RESULT")
    print(json.dumps(nested["pooled"], indent=2))
    print("replay_gate_passed:", nested["replay_gate_passed"])
    print("all_inner_selections_eligible:", nested["all_inner_selections_eligible"])
    print("gate_and_ranker_diagnostics:")
    print(json.dumps(nested["gate_and_ranker_diagnostics"], indent=2))

    print("\nDIAGNOSTIC UPPER BOUNDS")
    for name, result in nested["diagnostic_upper_bounds"].items():
        pooled = result["pooled"]
        print(
            name,
            "final_accuracy=", pooled["final_accuracy"],
            "gain=", pooled["absolute_gain"],
            "rescues=", pooled["rescued_count"],
            "damages=", pooled["damaged_count"],
        )

    outer_path = root / "nested_replay" / "nested_outer_results.csv"
    print("\nOUTER-SEED SELECTED CONFIGURATIONS")
    display(pd.read_csv(outer_path))

    print("\nBINDING PHASE-2 DECISION")
    print(json.dumps(decision, indent=2))

    required = [
        root / "decision.json",
        root / "hard_checks.json",
        root / "aggregate_summary.json",
        root / "capacity" / "capacity_summary.csv",
        root / "capacity" / "capacity_per_example_top3.csv",
        root / "nonlinear_feature_manifest.json",
        root / "nested_replay" / "nested_loso_summary.json",
        root / "nested_replay" / "nested_outer_results.csv",
        root / "nested_replay" / "nested_gate_ranker_diagnostics.csv",
    ]
    print("\nREQUIRED OUTPUTS")
    for path in required:
        print(path, "exists=", path.is_file())

## Outputs to retain for Phase 3

Retain the complete v0.23T run folder, especially:

- `decision.json`
- `hard_checks.json`
- `aggregate_summary.json`
- `capacity/capacity_summary.csv`
- `nested_replay/nested_loso_summary.json`
- `nested_replay/nested_outer_results.csv`
- `nested_replay/nested_gate_ranker_diagnostics.csv`
- `protected_probe/selected_protected_probe_veto.json` and `selected_verifier.joblib` only when replay selection passed.

These results close Phase 2 and determine whether Phase 3 carries forward the nonlinear verifier or introduces new upstream verifier supervision.